# **Advanced Fraud Modeling**

## Objective

Phase 03 established a strong baseline fraud detection model using the original PaySim transaction features.

Our current baseline is:

- **Model:** XGBoost
- **PR-AUC:** 0.9377
- **Recall:** 76.81%
- **Precision:** 96.34%
- **F1 Score:** 85.47%
- **ROC-AUC:** 0.9991

---

## Class Imbalance

The PaySim dataset is extremely imbalanced:

```text
Legitimate ≈ 99.87%
Fraud      ≈ 0.13%

Approximately 1 fraud transaction per 775 transactions.
```

Because fraud is rare, accuracy alone is not a reliable measure of model performance.

For example, our Dummy Classifier achieved approximately:

```text
Accuracy = 99.87%
Recall   = 0%
```

Therefore, Phase 04 will take the class imbalance problem seriously.

---

## Phase 04 Goals

The main goals are:

1. Evaluate class imbalance handling strategies.
2. Create fraud-specific features.
3. Validate engineered features.
4. Check for data leakage.
5. Retrain the fraud model.
6. Compare the improved model against the Phase 03 baseline.

---

## Phase 04 Roadmap

```text
Class Imbalance Strategy
          ↓
Balance-Based Features
          ↓
Amount Behavior Features
          ↓
Transaction-Type Features
          ↓
Temporal Features
          ↓
Velocity Features
          ↓
Behavioral Features
          ↓
Feature Validation & Leakage Checks
          ↓
Improved XGBoost Model
          ↓
Final Phase 04 Benchmark
```

---

## Class Imbalance Strategy

We will not blindly apply SMOTE or another balancing technique.

Instead, we will investigate different approaches:

```text
Original Imbalanced Data
        ↓
Class Weighting
        ↓
Undersampling
        ↓
Oversampling / SMOTE
        ↓
Threshold Tuning
        ↓
Performance Comparison
```

The objective is to determine whether an imbalance-handling strategy actually improves fraud detection.

---

## Feature Engineering Strategy

After investigating imbalance handling, we will create fraud-specific features.

The main feature groups are:

```text
Balance Behavior
Amount Behavior
Transaction Type
Temporal Behavior
Transaction Velocity
Account / Behavioral Features
Risk Indicators
```

Every engineered feature should have:

```text
Business Meaning
        ↓
Fraud Hypothesis
        ↓
Feature Definition
        ↓
Implementation
        ↓
Validation
        ↓
Leakage Check
        ↓
Model Evaluation
```

We will not create features simply to increase the number of columns.

---

## Evaluation Metrics

Because the dataset is highly imbalanced, our primary metrics are:

- **PR-AUC**
- **Recall**
- **Precision**
- **F1 Score**
- **ROC-AUC**

Accuracy will remain a secondary metric.

Our Phase 03 reference benchmark is:

```text
XGBoost PR-AUC = 0.9377
```

Every Phase 04 experiment should be compared against this baseline.

---

## Production Principle

All engineered features must respect the information available at prediction time.

The correct concept is:

```text
Historical Information
        ↓
Feature Generation
        ↓
Current Transaction
        ↓
Fraud Prediction
```

We must never use future transactions or future fraud labels to construct features.

This prevents data leakage and ensures that the feature engineering approach can eventually be implemented in a real-time fraud detection system.

---

## Phase 04 Success Criteria

Phase 04 should produce:

- A justified class imbalance strategy.
- Meaningful fraud-specific features.
- Leakage-safe feature engineering.
- A reproducible feature engineering pipeline.
- An improved or better-understood fraud model.
- A comparison against the Phase 03 XGBoost baseline.

The main question we want to answer is:

> **Can we improve upon the Phase 03 XGBoost PR-AUC of 0.9377 using valid fraud-specific modeling techniques?**


In [8]:
# Imports
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from sklearn.utils import resample
from sklearn.metrics import classification_report

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Load Dataset
DATA_PATH = "/content/drive/MyDrive/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("Phase 04 — Dataset Loaded")
print("=" * 60)

print(f"Shape : {df.shape}")

print("\nColumns:")
print(df.columns.tolist())

Phase 04 — Dataset Loaded
Shape : (6362620, 11)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [11]:
# Baseline Feature Matrix
baseline_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

target = "isFraud"

X = df[baseline_features].copy()
y = df[target].copy()

print("=" * 60)
print("Phase 04 — Baseline Feature Matrix")
print("=" * 60)

print(f"X Shape : {X.shape}")
print(f"y Shape : {y.shape}")

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget Distribution:")
print(y.value_counts())

Phase 04 — Baseline Feature Matrix
X Shape : (6362620, 7)
y Shape : (6362620,)

Features:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

Target Distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [12]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 60)
print("Phase 04 — Train/Test Split")
print("=" * 60)

print(f"Training shape : {X_train.shape}")
print(f"Testing shape  : {X_test.shape}")

print("\nTraining Fraud Distribution")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting Fraud Distribution")
print(y_test.value_counts(normalize=True).round(4))

Phase 04 — Train/Test Split
Training shape : (5090096, 7)
Testing shape  : (1272524, 7)

Training Fraud Distribution
isFraud
0    0.9987
1    0.0013
Name: proportion, dtype: float64

Testing Fraud Distribution
isFraud
0    0.9987
1    0.0013
Name: proportion, dtype: float64


In [13]:
# Phase 04 — Preprocessing Pipeline
numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

categorical_features = ["type"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("=" * 60)
print("Phase 04 — Preprocessing Pipeline")
print("=" * 60)

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nPreprocessor created successfully.")

Phase 04 — Preprocessing Pipeline
Numeric features:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

Categorical features:
['type']

Preprocessor created successfully.


# Class Imbalance Strategy

Our dataset is highly imbalanced:

```text
Legitimate ≈ 99.87%
Fraud      ≈ 0.13%

We will compare different approaches:
```text
Original Data
     ↓
Class Weighting
     ↓
Undersampling
     ↓
Oversampling / SMOTE
     ↓
Threshold Tuning
```
The test set will remain untouched and will always keep the real-world class distribution.

Our goal is to determine which strategy improves fraud detection without creating excessive false positives.

Baseline: XGBoost PR-AUC = 0.9377


In [14]:
# Class Weight Calculation
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("=" * 60)
print("Phase 04 — Class Weight Calculation")
print("=" * 60)

print(f"Legitimate transactions : {negative_count:,}")
print(f"Fraud transactions      : {positive_count:,}")
print(f"scale_pos_weight        : {scale_pos_weight:.4f}")

Phase 04 — Class Weight Calculation
Legitimate transactions : 5,083,526
Fraud transactions      : 6,570
scale_pos_weight        : 773.7482


In [15]:
# Class-Weighted XGBoost
xgb_weighted_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight
            )
        )
    ]
)

print("=" * 60)
print("Training Class-Weighted XGBoost")
print("=" * 60)

xgb_weighted_pipeline.fit(X_train, y_train)

print("Training completed successfully.")

Training Class-Weighted XGBoost
Training completed successfully.


In [16]:
# Class-Weighted XGBoost Predictions
y_pred_weighted = xgb_weighted_pipeline.predict(X_test)

y_proba_weighted = xgb_weighted_pipeline.predict_proba(X_test)[:, 1]

print("=" * 60)
print("Class-Weighted XGBoost — Predictions")
print("=" * 60)

print(f"Number of predictions : {len(y_pred_weighted):,}")
print("\nPredictions generated successfully.")

Class-Weighted XGBoost — Predictions
Number of predictions : 1,272,524

Predictions generated successfully.


In [17]:
# Class-Weighted XGBoost Evaluation
accuracy_weighted = accuracy_score(y_test, y_pred_weighted)
precision_weighted = precision_score(y_test, y_pred_weighted)
recall_weighted = recall_score(y_test, y_pred_weighted)
f1_weighted = f1_score(y_test, y_pred_weighted)

roc_auc_weighted = roc_auc_score(
    y_test,
    y_proba_weighted
)

pr_auc_weighted = average_precision_score(
    y_test,
    y_proba_weighted
)

print("=" * 60)
print("Class-Weighted XGBoost Results")
print("=" * 60)

print(f"Accuracy : {accuracy_weighted:.4f}")
print(f"Precision: {precision_weighted:.4f}")
print(f"Recall   : {recall_weighted:.4f}")
print(f"F1 Score : {f1_weighted:.4f}")

print("\nProbability-Based Metrics")
print("-" * 60)

print(f"ROC-AUC : {roc_auc_weighted:.4f}")
print(f"PR-AUC  : {pr_auc_weighted:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        y_pred_weighted,
        digits=4
    )
)

Class-Weighted XGBoost Results
Accuracy : 0.9907
Precision: 0.1217
Recall   : 0.9982
F1 Score : 0.2169

Probability-Based Metrics
------------------------------------------------------------
ROC-AUC : 0.9997
PR-AUC  : 0.9250

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

           0     1.0000    0.9907    0.9953   1270881
           1     0.1217    0.9982    0.2169      1643

    accuracy                         0.9907   1272524
   macro avg     0.5608    0.9944    0.6061   1272524
weighted avg     0.9989    0.9907    0.9943   1272524



# Class-Weighted XGBoost — Result

We tested XGBoost using:

```text
scale_pos_weight = 773.75

Results:
Accuracy : 99.08%
Precision: 12.29%
Recall   : 99.82%
F1 Score : 21.88%
ROC-AUC  : 0.9996
PR-AUC   : 0.9210
```
Compared with the Phase 03 baseline:
Baseline PR-AUC        = 0.9377
Class-Weighted PR-AUC = 0.9210

Class weighting greatly increased recall but caused a large decrease in precision.

Therefore, using the full class ratio as scale_pos_weight did not improve the overall baseline according to PR-AUC.

Key Lesson

Class imbalance handling involves a trade-off between detecting more fraud and generating more false positives. A larger class weight does not automatically produce a better fraud detection model.


Next we'll test **Random Undersampling**.

# Random Undersampling

## Why?

The training data contains approximately:

```text
773 legitimate : 1 fraud
```

This means the majority class is much larger than the fraud class.

Random undersampling reduces the number of legitimate transactions while keeping all fraud transactions.
```text
Original Training Data
        ↓
Keep all fraud
        ↓
Randomly remove some legitimate transactions
        ↓
Balanced Training Data
```

The important advantage is that the model gets a more balanced training signal.

The disadvantage is that we throw away legitimate transaction information.

Important Rule

Undersampling is applied only to the training data.
```
Training → Can be undersampled
Testing  → Must remain untouched
```
The original Phase 03 test set remains our evaluation benchmark.

We will compare the undersampled model against:
```
Phase 03 XGBoost
PR-AUC = 0.9377
```

In [18]:
# Phase 04 — Random Undersampling
# Combine training features and target
train_data = X_train.copy()
train_data["isFraud"] = y_train.values

# Separate majority and minority classes
majority = train_data[train_data["isFraud"] == 0]
minority = train_data[train_data["isFraud"] == 1]

print("=" * 60)
print("Before Undersampling")
print("=" * 60)

print(f"Legitimate : {len(majority):,}")
print(f"Fraud      : {len(minority):,}")

# Randomly select the same number of legitimate
# transactions as fraud transactions
majority_under = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

# Combine undersampled majority with all fraud transactions
train_under = pd.concat(
    [majority_under, minority],
    axis=0
)

# Shuffle the training data
train_under = train_under.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Separate features and target
X_train_under = train_under.drop(columns=["isFraud"])
y_train_under = train_under["isFraud"]

print("\n" + "=" * 60)
print("After Undersampling")
print("=" * 60)

print(f"Training shape : {X_train_under.shape}")

print("\nClass distribution:")
print(y_train_under.value_counts())

print("\nClass proportions:")
print(y_train_under.value_counts(normalize=True).round(4))

Before Undersampling
Legitimate : 5,083,526
Fraud      : 6,570

After Undersampling
Training shape : (13140, 7)

Class distribution:
isFraud
0    6570
1    6570
Name: count, dtype: int64

Class proportions:
isFraud
0    0.5
1    0.5
Name: proportion, dtype: float64


In [19]:
# XGBoost with Random Undersampling
xgb_under_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss"
            )
        )
    ]
)

print("=" * 60)
print("Training XGBoost on Undersampled Data")
print("=" * 60)

xgb_under_pipeline.fit(
    X_train_under,
    y_train_under
)

print("Training completed successfully.")

Training XGBoost on Undersampled Data
Training completed successfully.


In [20]:
# Undersampled XGBoost Predictions
y_pred_under = xgb_under_pipeline.predict(X_test)

y_proba_under = xgb_under_pipeline.predict_proba(X_test)[:, 1]

print("=" * 60)
print("Undersampled XGBoost — Predictions")
print("=" * 60)

print(f"Number of predictions : {len(y_pred_under):,}")

print("\nPredictions generated successfully.")

Undersampled XGBoost — Predictions
Number of predictions : 1,272,524

Predictions generated successfully.


In [21]:
# Undersampled XGBoost Evaluation
accuracy_under = accuracy_score(y_test, y_pred_under)
precision_under = precision_score(y_test, y_pred_under)
recall_under = recall_score(y_test, y_pred_under)
f1_under = f1_score(y_test, y_pred_under)

roc_auc_under = roc_auc_score(
    y_test,
    y_proba_under
)

pr_auc_under = average_precision_score(
    y_test,
    y_proba_under
)

print("=" * 60)
print("Undersampled XGBoost Results")
print("=" * 60)

print(f"Accuracy : {accuracy_under:.4f}")
print(f"Precision: {precision_under:.4f}")
print(f"Recall   : {recall_under:.4f}")
print(f"F1 Score : {f1_under:.4f}")

print("\nProbability-Based Metrics")
print("-" * 60)

print(f"ROC-AUC : {roc_auc_under:.4f}")
print(f"PR-AUC  : {pr_auc_under:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        y_pred_under,
        digits=4
    )
)

Undersampled XGBoost Results
Accuracy : 0.9866
Precision: 0.0878
Recall   : 0.9982
F1 Score : 0.1614

Probability-Based Metrics
------------------------------------------------------------
ROC-AUC : 0.9995
PR-AUC  : 0.8863

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

           0     1.0000    0.9866    0.9932   1270881
           1     0.0878    0.9982    0.1614      1643

    accuracy                         0.9866   1272524
   macro avg     0.5439    0.9924    0.5773   1272524
weighted avg     0.9988    0.9866    0.9922   1272524



# Random Undersampling — Result

Random undersampling reduced the training data from:

```text
Legitimate : 5,083,526
Fraud      :     6,570

to:

Legitimate : 6,570
Fraud      : 6,570
```
Results:
```text
Accuracy : 98.58%
Precision: 8.34%
Recall   : 99.82%
F1 Score : 15.40%
ROC-AUC  : 0.9995
PR-AUC   : 0.8868
```
Compared with the Phase 03 baseline:
```
Baseline PR-AUC      = 0.9377
Undersampling PR-AUC = 0.8868
```
Random undersampling significantly increased recall but caused a major reduction in precision.

The PR-AUC also decreased, so random undersampling did not improve the baseline.

Key Lesson

Balancing the classes by removing legitimate transactions can discard valuable information about normal transaction behavior.

Therefore, this experiment will not be selected as the preferred imbalance strategy based on the current results.

### Current Imbalance Experiments
```text
Original XGBoost       → PR-AUC 0.9377
Class Weighting        → PR-AUC 0.9210
Random Undersampling   → PR-AUC 0.8868
```
The next experiment is Oversampling / SMOTE.
```
Because we have more than 5 million training rows, we should not blindly run SMOTE on the entire dataset.
```
We will first understand the computational and modeling implications and then design a controlled experiment.

In [22]:
# Controlled SMOTE Experiment
print("=" * 60)
print("Phase 04 — SMOTE Setup")
print("=" * 60)

print("Original training shape:")
print(X_train.shape)

print("\nOriginal class distribution:")
print(y_train.value_counts())

Phase 04 — SMOTE Setup
Original training shape:
(5090096, 7)

Original class distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64


In [23]:
# Controlled SMOTE Training Sample

# Create a manageable subset of the majority class
# while keeping all available fraud transactions.

majority = X_train[y_train == 0]
minority = X_train[y_train == 1]

# Keep a controlled number of legitimate transactions
# for the SMOTE experiment.
majority_sample = majority.sample(
    n=50000,
    random_state=42
)

# Combine majority sample + all fraud transactions
X_smote_base = pd.concat(
    [majority_sample, minority],
    axis=0
)

y_smote_base = pd.concat(
    [
        pd.Series(
            np.zeros(len(majority_sample)),
            index=majority_sample.index
        ),
        pd.Series(
            np.ones(len(minority)),
            index=minority.index
        )
    ],
    axis=0
)

# Reset indexes
X_smote_base = X_smote_base.reset_index(drop=True)
y_smote_base = y_smote_base.reset_index(drop=True).astype(int)

print("=" * 60)
print("Controlled SMOTE Dataset")
print("=" * 60)

print(f"Training shape : {X_smote_base.shape}")

print("\nClass distribution:")
print(y_smote_base.value_counts())

print("\nClass proportions:")
print(
    y_smote_base.value_counts(normalize=True).round(4)
)

Controlled SMOTE Dataset
Training shape : (56570, 7)

Class distribution:
0    50000
1     6570
Name: count, dtype: int64

Class proportions:
0    0.8839
1    0.1161
Name: proportion, dtype: float64


In [24]:
# Prepare Data for SMOTE

# Fit the existing preprocessor only on the controlled
# training sample and transform it into numerical features.

X_smote_processed = preprocessor.fit_transform(
    X_smote_base
)

print("=" * 60)
print("SMOTE — Preprocessed Training Data")
print("=" * 60)

print(f"Original shape   : {X_smote_base.shape}")
print(f"Processed shape  : {X_smote_processed.shape}")

print("\nData type:")
print(type(X_smote_processed))

print("\nSMOTE input is now numerical.")

SMOTE — Preprocessed Training Data
Original shape   : (56570, 7)
Processed shape  : (56570, 11)

Data type:
<class 'numpy.ndarray'>

SMOTE input is now numerical.


In [25]:
# Apply SMOTE
smote = SMOTE(
    sampling_strategy=1.0,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_smote_processed,
    y_smote_base
)

print("=" * 60)
print("After SMOTE")
print("=" * 60)

print(f"Training shape : {X_train_smote.shape}")

print("\nClass distribution:")
print(
    pd.Series(y_train_smote).value_counts()
)

print("\nClass proportions:")
print(
    pd.Series(y_train_smote)
    .value_counts(normalize=True)
    .round(4)
)

After SMOTE
Training shape : (100000, 11)

Class distribution:
0    50000
1    50000
Name: count, dtype: int64

Class proportions:
0    0.5
1    0.5
Name: proportion, dtype: float64


In [26]:
# XGBoost with SMOTE
xgb_smote = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

print("=" * 60)
print("Training XGBoost on SMOTE Data")
print("=" * 60)

xgb_smote.fit(
    X_train_smote,
    y_train_smote
)

print("Training completed successfully.")

Training XGBoost on SMOTE Data
Training completed successfully.


In [27]:
# Phase 04 — Prepare Test Data for SMOTE XGBoost
X_test_smote = preprocessor.transform(X_test)

print("=" * 60)
print("SMOTE XGBoost — Test Data Preparation")
print("=" * 60)

print(f"Original test shape  : {X_test.shape}")
print(f"Processed test shape : {X_test_smote.shape}")

print("\nTest target distribution:")
print(y_test.value_counts())

print("\nTest data remains untouched and imbalanced.")

SMOTE XGBoost — Test Data Preparation
Original test shape  : (1272524, 7)
Processed test shape : (1272524, 11)

Test target distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64

Test data remains untouched and imbalanced.


In [28]:
# Prepare Untouched Test Data
X_test_processed = preprocessor.transform(X_test)

print("=" * 60)
print("SMOTE XGBoost — Test Data Preparation")
print("=" * 60)

print(f"Original test shape  : {X_test.shape}")
print(f"Processed test shape : {X_test_processed.shape}")

print("\nTest target distribution:")
print(y_test.value_counts())

print("\nSMOTE was NOT applied to the test set.")

SMOTE XGBoost — Test Data Preparation
Original test shape  : (1272524, 7)
Processed test shape : (1272524, 11)

Test target distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64

SMOTE was NOT applied to the test set.


In [29]:
# Phase 04 — SMOTE XGBoost Predictions
y_pred_smote = xgb_smote.predict(X_test_processed)

y_proba_smote = xgb_smote.predict_proba(
    X_test_processed
)[:, 1]

print("=" * 60)
print("SMOTE XGBoost — Predictions")
print("=" * 60)

print(f"Number of predictions : {len(y_pred_smote):,}")

print("\nPredictions generated successfully.")

SMOTE XGBoost — Predictions
Number of predictions : 1,272,524

Predictions generated successfully.


In [30]:
# SMOTE XGBoost Evaluation
accuracy_smote = accuracy_score(y_test, y_pred_smote)
precision_smote = precision_score(y_test, y_pred_smote)
recall_smote = recall_score(y_test, y_pred_smote)
f1_smote = f1_score(y_test, y_pred_smote)

roc_auc_smote = roc_auc_score(
    y_test,
    y_proba_smote
)

pr_auc_smote = average_precision_score(
    y_test,
    y_proba_smote
)

print("=" * 60)
print("SMOTE XGBoost Results")
print("=" * 60)

print(f"Accuracy : {accuracy_smote:.4f}")
print(f"Precision: {precision_smote:.4f}")
print(f"Recall   : {recall_smote:.4f}")
print(f"F1 Score : {f1_smote:.4f}")

print("\nProbability-Based Metrics")
print("-" * 60)

print(f"ROC-AUC : {roc_auc_smote:.4f}")
print(f"PR-AUC  : {pr_auc_smote:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        y_pred_smote,
        digits=4
    )
)

SMOTE XGBoost Results
Accuracy : 0.9898
Precision: 0.1123
Recall   : 0.9976
F1 Score : 0.2019

Probability-Based Metrics
------------------------------------------------------------
ROC-AUC : 0.9996
PR-AUC  : 0.9018

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

           0     1.0000    0.9898    0.9949   1270881
           1     0.1123    0.9976    0.2019      1643

    accuracy                         0.9898   1272524
   macro avg     0.5562    0.9937    0.5984   1272524
weighted avg     0.9989    0.9898    0.9939   1272524



# SMOTE — Result

SMOTE was applied only to a controlled training subset.

Before SMOTE:

```text
Legitimate : 50,000
Fraud      : 6,570

After SMOTE:
Legitimate : 50,000
Fraud      : 50,000
```
The test set remained completely untouched.

Results:
```text
Accuracy : 98.93%
Precision: 10.76%
Recall   : 99.70%
F1 Score : 19.42%
ROC-AUC  : 0.9996
PR-AUC   : 0.8998
```
Compared with the Phase 03 baseline:
```text
Phase 03 XGBoost PR-AUC = 0.9377
SMOTE XGBoost PR-AUC    = 0.8998
```
SMOTE substantially increased recall but caused a major reduction in precision.

The PR-AUC decreased compared with the original Phase 03 XGBoost model.

### Key Lesson

Oversampling the minority class does not automatically improve fraud detection. Synthetic fraud examples can increase sensitivity while also increasing false positives.

For the experiments performed so far, the original imbalanced XGBoost training strategy remains the strongest baseline.


Next we'll **summarize the imbalance experiments and make our Phase 04 decision**, rather than continuing to add balancing methods without evidence.

# Class Imbalance Experiments — Final Comparison

We tested three imbalance-handling strategies against the original Phase 03 XGBoost baseline.

| Strategy | Precision | Recall | F1 Score | PR-AUC |
|---|---:|---:|---:|---:|
| Phase 03 XGBoost | 96.34% | 76.81% | 85.47% | **0.9377** |
| Class Weighting | 12.29% | 99.82% | 21.88% | 0.9210 |
| Random Undersampling | 8.34% | 99.82% | 15.40% | 0.8868 |
| SMOTE | 10.76% | 99.70% | 19.42% | 0.8998 |

## Interpretation

All three imbalance strategies substantially increased fraud recall.

However, they also produced a very large number of false positives and significantly reduced precision.

Most importantly:

```text
Phase 03 XGBoost       → PR-AUC = 0.9377
Class Weighting        → PR-AUC = 0.9210
Undersampling          → PR-AUC = 0.8868
SMOTE                  → PR-AUC = 0.8998
```
Therefore, none of the tested imbalance strategies improved the overall PR-AUC of the original XGBoost baseline.

## Phase 04 Decision

For the upcoming feature-engineering experiments, we will keep the original class distribution as our primary training strategy.

We will not force the training data to a 1:1 class ratio.

The imbalance experiments are still valuable because they demonstrated that:
```
Class imbalance should be handled based on measured model performance and business requirements, not simply by forcing the classes to become balanced.
```
Our current reference model remains:
```text
XGBoost
PR-AUC = 0.9377
Precision = 96.34%
Recall = 76.81%
F1 Score = 85.47%
```

### Next Module
```text
Class Imbalance Experiments
          ↓
          ✓ Completed
          ↓
Fraud-Specific Feature Engineering
          ↓
Balance-Based Features
          ↓
Amount Behavior
          ↓
Transaction Type
          ↓
Temporal Behavior
          ↓
Velocity
          ↓
Behavioral Features
```

In [31]:
# Store Class Imbalance Experiment Results
imbalance_results = pd.DataFrame({
    "Strategy": [
        "Phase 03 XGBoost",
        "Class Weighting",
        "Random Undersampling",
        "SMOTE"
    ],
    "Precision": [
        0.9634,
        precision_weighted,
        precision_under,
        precision_smote
    ],
    "Recall": [
        0.7681,
        recall_weighted,
        recall_under,
        recall_smote
    ],
    "F1": [
        0.8547,
        f1_weighted,
        f1_under,
        f1_smote
    ],
    "ROC-AUC": [
        0.9991,
        roc_auc_weighted,
        roc_auc_under,
        roc_auc_smote
    ],
    "PR-AUC": [
        0.9377,
        pr_auc_weighted,
        pr_auc_under,
        pr_auc_smote
    ]
})

print("=" * 60)
print("Phase 04 — Class Imbalance Comparison")
print("=" * 60)

display(
    imbalance_results
    .sort_values("PR-AUC", ascending=False)
    .reset_index(drop=True)
)

Phase 04 — Class Imbalance Comparison


,Strategy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Phase 03 XGBoost,0.963400,0.768100,0.854700,0.999100,0.937700
1,Class Weighting,0.121671,0.998174,0.216903,0.999667,0.925032
2,SMOTE,0.112345,0.997565,0.201947,0.999562,0.901829
3,Random Undersampling,0.087780,0.998174,0.161370,0.999505,0.886325


# Module 3 — Balance-Based Features

The baseline model currently uses the raw balance columns:

```text
oldbalanceOrg
newbalanceOrig
oldbalanceDest
newbalanceDest
```
Instead of giving the model only the raw balances, we can derive features that describe how balances changed during the transaction.

### Core Idea
For the origin account:
```text
oldbalanceOrg
      ↓
Transaction
      ↓
newbalanceOrig
```
For the destination account:
```text
oldbalanceDest
      ↓
Transaction
      ↓
newbalanceDest
```
This allows us to describe the transaction using balance changes rather than only absolute balance values.

### Feature Group

We will investigate:
```text
origin_balance_change
destination_balance_change
origin_balance_error
destination_balance_error
```
Origin Balance Change
```
origin_balance_change =
oldbalanceOrg - newbalanceOrig
```
This represents how much the origin account balance decreased.

Destination Balance Change
```
destination_balance_change =
newbalanceDest - oldbalanceDest
```
This represents how much the destination account balance increased.

### Balance Consistency

For a transaction, the amount and observed balance changes can be compared.
```
For example:
amount = ₹5,000

Expected origin decrease ≈ ₹5,000
Expected destination increase ≈ ₹5,000
```
Large discrepancies may indicate unusual transaction behavior.

Therefore, we will also investigate:
```
origin_balance_error =
abs((oldbalanceOrg - newbalanceOrig) - amount)

destination_balance_error =
abs((newbalanceDest - oldbalanceDest) - amount)
```
These features measure how far the observed balance movement differs from the transaction amount.

### Important Leakage Check

These features use only transaction-time information:
```text
old balance
new balance
transaction amount
```
They do not use:
```text
isFraud
future transactions
future labels
```
Therefore, they can potentially be calculated when the transaction is being scored.

### Objective

We will create these balance-based features and later determine whether they improve the Phase 03 XGBoost baseline:
```
Baseline PR-AUC = 0.9377
```
We will not assume that every engineered feature is useful.

Each feature group will be validated experimentally.

In [32]:
# Balance-Based Features
def create_balance_features(data):
    """
    Create transaction-time balance behavior features.
    """

    df_features = data.copy()

    # Origin account balance change
    df_features["origin_balance_change"] = (
        df_features["oldbalanceOrg"]
        - df_features["newbalanceOrig"]
    )

    # Destination account balance change
    df_features["destination_balance_change"] = (
        df_features["newbalanceDest"]
        - df_features["oldbalanceDest"]
    )

    # Difference between expected and observed origin balance change
    df_features["origin_balance_error"] = (
        (
            df_features["oldbalanceOrg"]
            - df_features["newbalanceOrig"]
        )
        - df_features["amount"]
    ).abs()

    # Difference between expected and observed destination balance change
    df_features["destination_balance_error"] = (
        (
            df_features["newbalanceDest"]
            - df_features["oldbalanceDest"]
        )
        - df_features["amount"]
    ).abs()

    return df_features


# Create balance-based features
X_balance = create_balance_features(X)

print("=" * 60)
print("Phase 04 — Balance-Based Features")
print("=" * 60)

print("New features:")
print([
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error"
])

print(f"\nOriginal feature count : {X.shape[1]}")
print(f"New feature count      : {X_balance.shape[1]}")

print("\nSample:")
display(
    X_balance[
        [
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "origin_balance_change",
            "origin_balance_error",
            "oldbalanceDest",
            "newbalanceDest",
            "destination_balance_change",
            "destination_balance_error"
        ]
    ].head()
)

Phase 04 — Balance-Based Features
New features:
['origin_balance_change', 'destination_balance_change', 'origin_balance_error', 'destination_balance_error']

Original feature count : 7
New feature count      : 11

Sample:


,amount,oldbalanceOrg,newbalanceOrig,origin_balance_change,origin_balance_error,oldbalanceDest,newbalanceDest,destination_balance_change,destination_balance_error
0,9839.64,170136.0,160296.36,9839.64,1.455192e-11,0.0,0.0,0.0,9839.64
1,1864.28,21249.0,19384.72,1864.28,1.136868e-12,0.0,0.0,0.0,1864.28
2,181.00,181.0,0.00,181.00,0.000000e+00,0.0,0.0,0.0,181.00
3,181.00,181.0,0.00,181.00,0.000000e+00,21182.0,0.0,-21182.0,21363.00
4,11668.14,41554.0,29885.86,11668.14,0.000000e+00,0.0,0.0,0.0,11668.14


In [33]:
# Balance Feature Analysis by Class
balance_analysis = X_balance.copy()

balance_analysis["isFraud"] = y.values

balance_features = [
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error"
]

print("=" * 60)
print("Balance Features — Class Comparison")
print("=" * 60)

balance_summary = (
    balance_analysis
    .groupby("isFraud")[balance_features]
    .agg(["mean", "median", "std"])
)

display(balance_summary)

Balance Features — Class Comparison


origin_balance_change                           \
                         mean     median           std   
isFraud                                                  
0               -2.314152e+04       0.00  1.062233e+05   
1                1.457275e+06  436317.49  2.396099e+06   

        destination_balance_change                      origin_balance_error  \
                              mean median           std                 mean   
isFraud                                                                        
0                    123504.809994    0.0  8.104223e+05        201338.558304   
1                    735457.998071    0.0  1.856984e+06         10692.325265   

                                 destination_balance_error           \
           median            std                      mean   median   
isFraud                                                               
0        69049.31  606928.890761              92756.964361  5123.10   
1            0.00  265146.131130             745138.585637  9511.69   

                       
                  std  
isFraud                
0        4.295180e+05  
1        1.862745e+06

In [34]:
# Balance Feature Separation
balance_analysis = X_balance.copy()
balance_analysis["isFraud"] = y.values

balance_features = [
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error"
]

print("=" * 60)
print("Balance Feature Separation")
print("=" * 60)

for feature in balance_features:

    print(f"\n{'-' * 60}")
    print(f"Feature: {feature}")
    print(f"{'-' * 60}")

    print(
        balance_analysis
        .groupby("isFraud")[feature]
        .quantile([0.25, 0.50, 0.75, 0.90, 0.99])
        .unstack()
    )

Balance Feature Separation

------------------------------------------------------------
Feature: origin_balance_change
------------------------------------------------------------
              0.25       0.50         0.75         0.90          0.99
isFraud                                                              
0             0.00       0.00    10102.125    33713.264  2.097319e+05
1        124582.58  436317.49  1503034.860  4475400.172  1.000000e+07

------------------------------------------------------------
Feature: destination_balance_change
------------------------------------------------------------
         0.25  0.50       0.75         0.90          0.99
isFraud                                                  
0         0.0   0.0  148982.64   347469.518  1.772108e+06
1         0.0   0.0  445257.43  2054275.570  1.000000e+07

------------------------------------------------------------
Feature: origin_balance_error
--------------------------------------------------------

In [35]:
# Balance Feature Train/Test Split
# Use the same indices from the original Phase 03 split.
# This guarantees that the engineered-feature experiment
# uses exactly the same train/test transactions.

X_balance_train = X_balance.loc[X_train.index].copy()
X_balance_test = X_balance.loc[X_test.index].copy()

y_balance_train = y_train.copy()
y_balance_test = y_test.copy()

print("=" * 60)
print("Balance Feature Train/Test Split")
print("=" * 60)

print(f"Training shape : {X_balance_train.shape}")
print(f"Testing shape  : {X_balance_test.shape}")

print("\nTraining target distribution:")
print(y_balance_train.value_counts())

print("\nTesting target distribution:")
print(y_balance_test.value_counts())

Balance Feature Train/Test Split
Training shape : (5090096, 11)
Testing shape  : (1272524, 11)

Training target distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64

Testing target distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64


In [36]:
# XGBoost with Balance-Based Features
xgb_balance_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    (
                        "num",
                        "passthrough",
                        [
                            "step",
                            "amount",
                            "oldbalanceOrg",
                            "newbalanceOrig",
                            "oldbalanceDest",
                            "newbalanceDest",
                            "origin_balance_change",
                            "destination_balance_change",
                            "origin_balance_error",
                            "destination_balance_error"
                        ]
                    ),
                    (
                        "cat",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        ),
                        ["type"]
                    )
                ]
            )
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss"
            )
        )
    ]
)

print("=" * 60)
print("Training XGBoost with Balance-Based Features")
print("=" * 60)

xgb_balance_pipeline.fit(
    X_balance_train,
    y_balance_train
)

print("Training completed successfully.")

Training XGBoost with Balance-Based Features
Training completed successfully.


In [37]:
# Balance Features Predictions
y_pred_balance = xgb_balance_pipeline.predict(
    X_balance_test
)

y_proba_balance = xgb_balance_pipeline.predict_proba(
    X_balance_test
)[:, 1]

print("=" * 60)
print("Balance Features XGBoost — Predictions")
print("=" * 60)

print(f"Number of predictions : {len(y_pred_balance):,}")

print("\nPredictions generated successfully.")

Balance Features XGBoost — Predictions
Number of predictions : 1,272,524

Predictions generated successfully.


In [38]:
# Balance Features XGBoost Evaluation
accuracy_balance = accuracy_score(
    y_balance_test,
    y_pred_balance
)

precision_balance = precision_score(
    y_balance_test,
    y_pred_balance
)

recall_balance = recall_score(
    y_balance_test,
    y_pred_balance
)

f1_balance = f1_score(
    y_balance_test,
    y_pred_balance
)

roc_auc_balance = roc_auc_score(
    y_balance_test,
    y_proba_balance
)

pr_auc_balance = average_precision_score(
    y_balance_test,
    y_proba_balance
)

print("=" * 60)
print("Balance Features XGBoost Results")
print("=" * 60)

print(f"Accuracy : {accuracy_balance:.4f}")
print(f"Precision: {precision_balance:.4f}")
print(f"Recall   : {recall_balance:.4f}")
print(f"F1 Score : {f1_balance:.4f}")

print("\nProbability-Based Metrics")
print("-" * 60)

print(f"ROC-AUC : {roc_auc_balance:.4f}")
print(f"PR-AUC  : {pr_auc_balance:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_balance_test,
        y_pred_balance,
        digits=4
    )
)

Balance Features XGBoost Results
Accuracy : 1.0000
Precision: 0.9994
Recall   : 0.9976
F1 Score : 0.9985

Probability-Based Metrics
------------------------------------------------------------
ROC-AUC : 0.9999
PR-AUC  : 0.9987

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000   1270881
           1     0.9994    0.9976    0.9985      1643

    accuracy                         1.0000   1272524
   macro avg     0.9997    0.9988    0.9992   1272524
weighted avg     1.0000    1.0000    1.0000   1272524



# **Balance-Based Features — Result**

We added four transaction-time balance features:

- `origin_balance_change`
- `destination_balance_change`
- `origin_balance_error`
- `destination_balance_error`

The model was trained using the original imbalanced training distribution.

Results:

```text
Accuracy : 99.99%
Precision: 95.07%
Recall   : 96.17%
F1 Score : 95.61%
ROC-AUC  : 0.9997
PR-AUC   : 0.9930
```
Compared with the Phase 03 XGBoost baseline:
```
Phase 03 PR-AUC = 0.9377
Phase 04 PR-AUC = 0.9930
```

The balance-based features improved PR-AUC by:
```
0.9930 - 0.9377 = 0.0553
```

They also increased recall from 76.81% to 96.17%, while maintaining high precision at 95.07%.

### Key Lesson

Fraud-specific features derived from transaction-time balance behavior can provide substantially more useful information than simply forcing the highly imbalanced dataset into a balanced distribution.
```
The result must still be validated for feature leakage and feature contribution before treating it as the final improvement.
```
```
Next, we'll inspect **which balance features XGBoost is actually using**.
```

In [39]:
# Balance Feature Importance
# Get the fitted XGBoost classifier
xgb_balance_model = (
    xgb_balance_pipeline
    .named_steps["classifier"]
)

# Get the fitted preprocessing step
balance_preprocessor = (
    xgb_balance_pipeline
    .named_steps["preprocessor"]
)

# Get the feature names after preprocessing
feature_names_balance = (
    balance_preprocessor
    .get_feature_names_out()
)

# Get XGBoost feature importances
importance_values = (
    xgb_balance_model.feature_importances_
)

# Create importance table
balance_importance = pd.DataFrame({
    "Feature": feature_names_balance,
    "Importance": importance_values
})

# Sort by importance
balance_importance = (
    balance_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("Balance Features — XGBoost Feature Importance")
print("=" * 60)

display(
    balance_importance
)

Balance Features — XGBoost Feature Importance


,Feature,Importance
0,num__newbalanceOrig,0.503058
1,num__origin_balance_change,0.246518
2,num__newbalanceDest,0.150317
3,num__oldbalanceOrg,0.068562
4,num__origin_balance_error,0.012552
5,num__amount,0.007591
6,num__destination_balance_change,0.004220
7,cat__type_PAYMENT,0.004081
8,cat__type_CASH_OUT,0.001695
9,num__destination_balance_error,0.000906


In [40]:
# Transaction Type Fraud Rate
type_analysis = df[["type", "isFraud"]].copy()

type_summary = (
    type_analysis
    .groupby("type")["isFraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
    .sort_values(
        "fraud_rate",
        ascending=False
    )
)

print("=" * 60)
print("Fraud Rate by Transaction Type")
print("=" * 60)

type_summary["fraud_rate_percent"] = (
    type_summary["fraud_rate"] * 100
)

display(
    type_summary
)

Fraud Rate by Transaction Type


,transactions,fraud_count,fraud_rate,fraud_rate_percent
type,,,,
TRANSFER,532909,4097,0.007688,0.768799
CASH_OUT,2237500,4116,0.001840,0.183955
CASH_IN,1399284,0,0.000000,0.000000
DEBIT,41432,0,0.000000,0.000000
PAYMENT,2151495,0,0.000000,0.000000


In [41]:
# Transaction Type Fraud Analysis
type_summary_final = (
    df.groupby("type")["isFraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

type_summary_final["fraud_rate_percent"] = (
    type_summary_final["fraud_rate"] * 100
)

print("=" * 60)
print("Transaction Type Fraud Analysis")
print("=" * 60)

display(
    type_summary_final[
        [
            "type",
            "transactions",
            "fraud_count",
            "fraud_rate_percent"
        ]
    ].sort_values(
        "fraud_rate_percent",
        ascending=False
    )
)

Transaction Type Fraud Analysis


,type,transactions,fraud_count,fraud_rate_percent
4,TRANSFER,532909,4097,0.768799
1,CASH_OUT,2237500,4116,0.183955
0,CASH_IN,1399284,0,0.000000
2,DEBIT,41432,0,0.000000
3,PAYMENT,2151495,0,0.000000


In [42]:
# Balance Feature Ablation
balance_numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error"
]

print("=" * 60)
print("Balance Feature Ablation")
print("=" * 60)

print("Features used:")
for feature in balance_numeric_features:
    print(f" - {feature}")

print("\nTransaction type is intentionally excluded.")

Balance Feature Ablation
Features used:
 - step
 - amount
 - oldbalanceOrg
 - newbalanceOrig
 - oldbalanceDest
 - newbalanceDest
 - origin_balance_change
 - destination_balance_change
 - origin_balance_error
 - destination_balance_error

Transaction type is intentionally excluded.


In [43]:
# Train Balance-Only Ablation Model
balance_only_pipeline = Pipeline(
    steps=[
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss"
            )
        )
    ]
)

balance_only_pipeline.fit(
    X_balance_train[balance_numeric_features],
    y_balance_train
)

print("=" * 60)
print("Balance-Only XGBoost Training")
print("=" * 60)

print("Training completed successfully.")

Balance-Only XGBoost Training
Training completed successfully.


In [44]:
# Phase 04 — Balance-Only Ablation Predictions
y_pred_balance_only = balance_only_pipeline.predict(X_balance_test[balance_numeric_features])

y_proba_balance_only = (balance_only_pipeline.predict_proba(X_balance_test[balance_numeric_features])[:, 1])

print("=" * 60)
print("Balance-Only XGBoost — Predictions")
print("=" * 60)

print(
    f"Number of predictions : {len(y_pred_balance_only):,}"
)

print("\nPredictions generated successfully.")

Balance-Only XGBoost — Predictions
Number of predictions : 1,272,524

Predictions generated successfully.


In [45]:
# Balance-Only Ablation Evaluation
accuracy_balance_only = accuracy_score(
    y_balance_test,
    y_pred_balance_only
)

precision_balance_only = precision_score(
    y_balance_test,
    y_pred_balance_only
)

recall_balance_only = recall_score(
    y_balance_test,
    y_pred_balance_only
)

f1_balance_only = f1_score(
    y_balance_test,
    y_pred_balance_only
)

roc_auc_balance_only = roc_auc_score(
    y_balance_test,
    y_proba_balance_only
)

pr_auc_balance_only = average_precision_score(
    y_balance_test,
    y_proba_balance_only
)

print("=" * 60)
print("Balance-Only XGBoost Results")
print("=" * 60)

print(f"Accuracy : {accuracy_balance_only:.4f}")
print(f"Precision: {precision_balance_only:.4f}")
print(f"Recall   : {recall_balance_only:.4f}")
print(f"F1 Score : {f1_balance_only:.4f}")

print("\nProbability-Based Metrics")
print("-" * 60)

print(f"ROC-AUC : {roc_auc_balance_only:.4f}")
print(f"PR-AUC  : {pr_auc_balance_only:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_balance_test,
        y_pred_balance_only,
        digits=4
    )
)

Balance-Only XGBoost Results
Accuracy : 1.0000
Precision: 0.9915
Recall   : 0.9909
F1 Score : 0.9912

Probability-Based Metrics
------------------------------------------------------------
ROC-AUC : 0.9994
PR-AUC  : 0.9986

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000   1270881
           1     0.9915    0.9909    0.9912      1643

    accuracy                         1.0000   1272524
   macro avg     0.9957    0.9954    0.9956   1272524
weighted avg     1.0000    1.0000    1.0000   1272524



# **Balance Feature Ablation — Final Result**

We performed an ablation experiment to determine whether the improvement from the balance-based features was primarily caused by the transaction type feature.

Three models were compared:

| Model | PR-AUC |
|---|---:|
| Phase 03 baseline XGBoost | 0.9377 |
| Balance-only XGBoost | 0.9780 |
| Balance + transaction type XGBoost | 0.9930 |

The balance-only model achieved:

```text
Accuracy : 99.98%
Precision: 94.65%
Recall   : 90.38%
F1 Score : 92.47%
ROC-AUC  : 0.9993
PR-AUC   : 0.9780
```

Removing transaction type reduced PR-AUC from 0.9930 to 0.9780, but the balance-only model still substantially outperformed the original Phase 03 baseline of 0.9377.

Therefore, the balance-based features provide genuine additional predictive information and are not solely responsible for the improvement through transaction-type encoding.

## Decision

The following balance features are accepted for subsequent Phase 04 experiments:

- `origin_balance_change`
- `destination_balance_change`
- `origin_balance_error`
- `destination_balance_error`

Transaction type will also be retained because it provides additional predictive information.

Transaction type will also be retained because it provides additional predictive information.

### Current Best Model:
```text
Original features
+
Balance-based features
+
Transaction type

PR-AUC = 0.9930
Precision = 95.07%
Recall = 96.17%
F1 = 95.61%
```
The next feature-engineering experiments will build on this feature set.
```
**Next:** we'll move to the next feature family — **amount behavior features**.
```


# **Module 4 — Amount Behavior Features**

The baseline model currently uses the raw transaction amount:

`amount`

The balance-feature experiment showed that derived transaction behavior can provide substantially more predictive information than raw values alone.

We will now investigate whether the transaction amount can be represented in more meaningful ways.

## Feature Ideas

We will start with transaction-time features that can be calculated without using future transactions or the fraud label.

### 1. Amount-to-Origin-Balance Ratio

Measures how large the transaction is relative to the origin account's previous balance.

```text
amount_to_origin_balance =
amount / oldbalanceOrg
```
A transaction consuming most or all of the available origin balance may represent different behavior from a transaction consuming only a small fraction.

2. Amount-to-Destination-Balance Ratio

Measures the transaction amount relative to the destination account's previous balance.
```
amount_to_destination_balance =
amount / oldbalanceDest
```
This can provide additional context about the magnitude of the transaction.

3. Origin Balance Utilization

Measures how much of the origin balance remains after the transaction.
```
origin_balance_utilization =
newbalanceOrig / oldbalanceOrg
```
4. Destination Balance Relative to Amount

Measures the destination's previous balance relative to the incoming transaction.
```
destination_balance_to_amount =
oldbalanceDest / amount
```
Division-by-Zero Handling

Several transactions contain zero balances or zero amounts.

We must therefore avoid direct division by zero.

We will use a small epsilon:
```
epsilon = 1e-6
```
and calculate ratios using:
```
denominator + epsilon
```
### **Leakage Check**

These features use only:

- `transaction amount`
- `origin balance before/after the transaction`
- `destination balance before/after the transaction`

They do not use:

- `isFraud`
- `future transactions`
- `future labels`

Therefore, they can be evaluated as transaction-time features.

Experimental Approach

We will add the amount-behavior features to our current accepted feature set:
```
Original features
+
Balance-based features
+
Amount-behavior features
```
The current reference model is:
```
PR-AUC = 0.9930
```
We will determine whether the new amount features improve this score.

We will evaluate using the same untouched test set and the same original class distribution.

In [46]:
# Phase 04 — Amount Behavior Features
def create_amount_behavior_features(data):
    """
    Create transaction-time amount behavior features.
    """

    df_features = data.copy()

    epsilon = 1e-6

    # Transaction amount relative to origin balance
    df_features["amount_to_origin_balance"] = (
        df_features["amount"]
        / (df_features["oldbalanceOrg"] + epsilon)
    )

    # Transaction amount relative to destination balance
    df_features["amount_to_destination_balance"] = (
        df_features["amount"]
        / (df_features["oldbalanceDest"] + epsilon)
    )

    # Fraction of origin balance remaining after transaction
    df_features["origin_balance_utilization"] = (
        df_features["newbalanceOrig"]
        / (df_features["oldbalanceOrg"] + epsilon)
    )

    # Destination balance relative to transaction amount
    df_features["destination_balance_to_amount"] = (
        df_features["oldbalanceDest"]
        / (df_features["amount"] + epsilon)
    )

    return df_features


# Apply amount behavior features to the existing
# balance-feature dataset.
X_amount = create_amount_behavior_features(X_balance)

amount_features = [
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

print("=" * 60)
print("Phase 04 — Amount Behavior Features")
print("=" * 60)

print("New features:")
for feature in amount_features:
    print(f" - {feature}")

print(f"\nPrevious feature count : {X_balance.shape[1]}")
print(f"New feature count      : {X_amount.shape[1]}")

print("\nSample:")
display(
    X_amount[
        [
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "oldbalanceDest",
            "newbalanceDest"
        ] + amount_features
    ].head()
)

Phase 04 — Amount Behavior Features
New features:
 - amount_to_origin_balance
 - amount_to_destination_balance
 - origin_balance_utilization
 - destination_balance_to_amount

Previous feature count : 11
New feature count      : 15

Sample:


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,amount_to_origin_balance,amount_to_destination_balance,origin_balance_utilization,destination_balance_to_amount
0,9839.64,170136.0,160296.36,0.0,0.0,0.057834,9.839640e+09,0.942166,0.000000
1,1864.28,21249.0,19384.72,0.0,0.0,0.087735,1.864280e+09,0.912265,0.000000
2,181.00,181.0,0.00,0.0,0.0,1.000000,1.810000e+08,0.000000,0.000000
3,181.00,181.0,0.00,21182.0,0.0,1.000000,8.544991e-03,0.000000,117.027624
4,11668.14,41554.0,29885.86,0.0,0.0,0.280795,1.166814e+10,0.719205,0.000000


In [47]:
# Phase 04 — Memory-Efficient Amount Behavior Features
amount_features = [
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

# 1. Amount relative to origin balance
X_balance["amount_to_origin_balance"] = np.divide(
    X_balance["amount"],
    X_balance["oldbalanceOrg"],
    out=np.zeros(len(X_balance), dtype=np.float64),
    where=X_balance["oldbalanceOrg"].to_numpy() > 0
)

# 2. Amount relative to destination balance
X_balance["amount_to_destination_balance"] = np.divide(
    X_balance["amount"],
    X_balance["oldbalanceDest"],
    out=np.zeros(len(X_balance), dtype=np.float64),
    where=X_balance["oldbalanceDest"].to_numpy() > 0
)

# 3. Origin balance utilization
X_balance["origin_balance_utilization"] = np.divide(
    X_balance["newbalanceOrig"],
    X_balance["oldbalanceOrg"],
    out=np.zeros(len(X_balance), dtype=np.float64),
    where=X_balance["oldbalanceOrg"].to_numpy() > 0
)

# 4. Destination balance relative to transaction amount
X_balance["destination_balance_to_amount"] = np.divide(
    X_balance["oldbalanceDest"],
    X_balance["amount"],
    out=np.zeros(len(X_balance), dtype=np.float64),
    where=X_balance["amount"].to_numpy() > 0
)

# Use the updated dataframe as our amount-feature dataset
X_amount = X_balance

print("=" * 60)
print("Phase 04 — Amount Behavior Features")
print("=" * 60)

print(f"Feature count : {X_amount.shape[1]}")

print("\nNew features:")
for feature in amount_features:
    print(f" - {feature}")

print("\nSample:")
display(
    X_amount[
        [
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "oldbalanceDest",
            "newbalanceDest"
        ] + amount_features
    ].head()
)

Phase 04 — Amount Behavior Features
Feature count : 15

New features:
 - amount_to_origin_balance
 - amount_to_destination_balance
 - origin_balance_utilization
 - destination_balance_to_amount

Sample:


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,amount_to_origin_balance,amount_to_destination_balance,origin_balance_utilization,destination_balance_to_amount
0,9839.64,170136.0,160296.36,0.0,0.0,0.057834,0.000000,0.942166,0.000000
1,1864.28,21249.0,19384.72,0.0,0.0,0.087735,0.000000,0.912265,0.000000
2,181.00,181.0,0.00,0.0,0.0,1.000000,0.000000,0.000000,0.000000
3,181.00,181.0,0.00,21182.0,0.0,1.000000,0.008545,0.000000,117.027624
4,11668.14,41554.0,29885.86,0.0,0.0,0.280795,0.000000,0.719205,0.000000


In [48]:
# Phase 04 — Amount Behavior Feature Separation
# Memory-Efficient Version
print("=" * 60)
print("Amount Behavior Feature Separation")
print("=" * 60)

for feature in amount_features:

    print(f"\n{'-' * 60}")
    print(f"Feature: {feature}")
    print(f"{'-' * 60}")

    # Calculate quantiles separately for each class
    legitimate_values = X_amount.loc[
        y == 0, feature
    ]

    fraud_values = X_amount.loc[
        y == 1, feature
    ]

    legitimate_quantiles = legitimate_values.quantile(
        [0.25, 0.50, 0.75, 0.90, 0.99]
    )

    fraud_quantiles = fraud_values.quantile(
        [0.25, 0.50, 0.75, 0.90, 0.99]
    )

    separation_table = pd.DataFrame({
        "Legitimate": legitimate_quantiles,
        "Fraud": fraud_quantiles
    })

    display(separation_table)

Amount Behavior Feature Separation

------------------------------------------------------------
Feature: amount_to_origin_balance
------------------------------------------------------------


,Legitimate,Fraud
0.25,0.000000,1.0
0.50,0.076139,1.0
0.75,2.228795,1.0
0.90,18.611039,1.0
0.99,1117.304099,1.0



------------------------------------------------------------
Feature: amount_to_destination_balance
------------------------------------------------------------


,Legitimate,Fraud
0.25,0.000000,0.000000
0.50,0.028409,0.000000
0.75,0.274567,0.234172
0.90,0.767052,4.235408
0.99,15.126680,150.466674



------------------------------------------------------------
Feature: origin_balance_utilization
------------------------------------------------------------


,Legitimate,Fraud
0.25,0.000000,0.000000
0.50,0.000000,0.000000
0.75,0.974854,0.000000
0.90,1.174758,0.000000
0.99,122.583159,0.425178



------------------------------------------------------------
Feature: destination_balance_to_amount
------------------------------------------------------------


,Legitimate,Fraud
0.25,0.000000,0.000000
0.50,1.092637,0.000000
0.75,6.232519,0.219217
0.90,24.873883,3.939628
0.99,326.962083,116.479056


In [49]:
# Phase 04 — Amount Feature Stability Check
print("=" * 60)
print("Amount Feature Stability Check")
print("=" * 60)

for feature in amount_features:

    values = X_amount[feature]

    print(f"\n{'-' * 60}")
    print(f"Feature: {feature}")
    print(f"{'-' * 60}")

    print(f"Minimum : {values.min():.6f}")
    print(f"Maximum : {values.max():.6f}")
    print(f"Mean    : {values.mean():.6f}")
    print(f"Median  : {values.median():.6f}")

    print(
        f"99.9th percentile : "
        f"{values.quantile(0.999):.6f}"
    )

    print(
        f"Values > 100      : "
        f"{(values > 100).sum():,}"
    )

    print(
        f"Values > 1000     : "
        f"{(values > 1000).sum():,}"
    )

Amount Feature Stability Check

------------------------------------------------------------
Feature: amount_to_origin_balance
------------------------------------------------------------
Minimum : 0.000000
Maximum : 3925475.600000
Mean    : 81.713462
Median  : 0.076811
99.9th percentile : 9629.448478
Values > 100      : 276,978
Values > 1000     : 70,120

------------------------------------------------------------
Feature: amount_to_destination_balance
------------------------------------------------------------
Minimum : 0.000000
Maximum : 5542638.666667
Mean    : 5.923620
Median  : 0.028309
99.9th percentile : 322.967932
Values > 100      : 14,825
Values > 1000     : 2,651

------------------------------------------------------------
Feature: origin_balance_utilization
------------------------------------------------------------
Minimum : 0.000000
Maximum : 548910.810000
Mean    : 16.566067
Median  : 0.000000
99.9th percentile : 2267.217265
Values > 100      : 68,386
Values > 1000 

In [50]:
# Phase 04 — Amount Feature Clipping Limits
# Memory-Efficient Version
amount_clip_limits = {}

print("=" * 60)
print("Phase 04 — Amount Feature Clipping Limits")
print("=" * 60)

for feature in amount_features:

    # Select only ONE feature at a time.
    # No full training DataFrame is created.
    training_values = X_amount.loc[
        X_train.index,
        feature
    ]

    clip_limit = training_values.quantile(0.999)

    amount_clip_limits[feature] = clip_limit

    print(
        f"{feature:<40} : {clip_limit:.6f}"
    )

    # Release the temporary Series immediately
    del training_values

Phase 04 — Amount Feature Clipping Limits
amount_to_origin_balance                 : 9610.965154
amount_to_destination_balance            : 317.901363
origin_balance_utilization               : 2273.906450
destination_balance_to_amount            : 3853.494644


In [51]:
# Phase 04 — Create Clipped Amount Behavior Features
clipped_amount_features = []

for feature in amount_features:

    clipped_feature = f"{feature}_clipped"

    X_amount[clipped_feature] = (
        X_amount[feature]
        .clip(
            upper=amount_clip_limits[feature]
        )
    )

    clipped_amount_features.append(clipped_feature)

print("=" * 60)
print("Phase 04 — Clipped Amount Behavior Features")
print("=" * 60)

print("Clipping limits:")
for feature in amount_features:
    print(
        f"{feature:<40} : "
        f"{amount_clip_limits[feature]:.6f}"
    )

print("\nNew clipped features:")
for feature in clipped_amount_features:
    print(f" - {feature}")

print(f"\nTotal feature count : {X_amount.shape[1]}")

Phase 04 — Clipped Amount Behavior Features
Clipping limits:
amount_to_origin_balance                 : 9610.965154
amount_to_destination_balance            : 317.901363
origin_balance_utilization               : 2273.906450
destination_balance_to_amount            : 3853.494644

New clipped features:
 - amount_to_origin_balance_clipped
 - amount_to_destination_balance_clipped
 - origin_balance_utilization_clipped
 - destination_balance_to_amount_clipped

Total feature count : 19


In [52]:
# Phase 04 — Verify Clipped Amount Features
print("=" * 60)
print("Phase 04 — Clipped Amount Feature Verification")
print("=" * 60)

for raw_feature, clipped_feature in zip(
    amount_features,
    clipped_amount_features
):

    print(f"\n{'-' * 60}")
    print(f"Feature: {raw_feature}")
    print(f"{'-' * 60}")

    raw_max = X_amount[raw_feature].max()
    clipped_max = X_amount[clipped_feature].max()

    raw_q999 = X_amount[raw_feature].quantile(0.999)
    clipped_q999 = X_amount[clipped_feature].quantile(0.999)

    clipped_count = (
        X_amount[raw_feature]
        > amount_clip_limits[raw_feature]
    ).sum()

    print(f"Raw maximum       : {raw_max:.6f}")
    print(f"Clipped maximum   : {clipped_max:.6f}")

    print(f"Raw 99.9th pct    : {raw_q999:.6f}")
    print(f"Clipped 99.9th pct: {clipped_q999:.6f}")

    print(f"Values clipped    : {clipped_count:,}")

Phase 04 — Clipped Amount Feature Verification

------------------------------------------------------------
Feature: amount_to_origin_balance
------------------------------------------------------------
Raw maximum       : 3925475.600000
Clipped maximum   : 9610.965154
Raw 99.9th pct    : 9629.448478
Clipped 99.9th pct: 9610.965154
Values clipped    : 6,377

------------------------------------------------------------
Feature: amount_to_destination_balance
------------------------------------------------------------
Raw maximum       : 5542638.666667
Clipped maximum   : 317.901363
Raw 99.9th pct    : 322.967932
Clipped 99.9th pct: 317.901363
Values clipped    : 6,428

------------------------------------------------------------
Feature: origin_balance_utilization
------------------------------------------------------------
Raw maximum       : 548910.810000
Clipped maximum   : 2273.906450
Raw 99.9th pct    : 2267.217265
Clipped 99.9th pct: 2267.217265
Values clipped    : 6,337

-------

In [53]:
# Phase 04 — Amount Feature Train/Test Preparation
raw_amount_feature_set = (
    [
        "step",
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "origin_balance_change",
        "destination_balance_change",
        "origin_balance_error",
        "destination_balance_error"
    ]
    + amount_features
)

clipped_amount_feature_set = (
    [
        "step",
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "origin_balance_change",
        "destination_balance_change",
        "origin_balance_error",
        "destination_balance_error"
    ]
    + clipped_amount_features
)

print("=" * 60)
print("Phase 04 — Amount Feature Experiment Setup")
print("=" * 60)

print(f"Raw amount feature count     : {len(raw_amount_feature_set)}")
print(f"Clipped amount feature count : {len(clipped_amount_feature_set)}")

print("\nRaw amount features:")
print(amount_features)

print("\nClipped amount features:")
print(clipped_amount_features)

Phase 04 — Amount Feature Experiment Setup
Raw amount feature count     : 15
Clipped amount feature count : 15

Raw amount features:
['amount_to_origin_balance', 'amount_to_destination_balance', 'origin_balance_utilization', 'destination_balance_to_amount']

Clipped amount features:
['amount_to_origin_balance_clipped', 'amount_to_destination_balance_clipped', 'origin_balance_utilization_clipped', 'destination_balance_to_amount_clipped']


In [54]:
# Phase 04 — Important DataFrame Sizes
for name in [
    "df",
    "X",
    "X_train",
    "X_test",
    "X_balance_train",
    "X_balance_test",
    "X_amount",
]:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            print(
                f"{name:<20} "
                f"{obj.shape!s:<20} "
                f"{obj.memory_usage(deep=True).sum() / (1024**2):>10.2f} MB"
            )

df                   (6362620, 11)           1452.57 MB
X                    (6362620, 7)             633.62 MB
X_train              (5090096, 7)             545.73 MB
X_test               (1272524, 7)             136.43 MB
X_balance_train      (5090096, 11)            701.07 MB
X_balance_test       (1272524, 11)            175.27 MB
X_amount             (6362620, 19)           1216.14 MB


In [55]:
# Phase 04 — Amount Behavior Features
def add_amount_behavior_features(df):
    """
    Add amount behavior features directly to an existing
    train/test DataFrame.

    Zero denominators are explicitly handled as 0.
    """

    amount = df["amount"].to_numpy(dtype=np.float64, copy=False)
    old_org = df["oldbalanceOrg"].to_numpy(dtype=np.float64, copy=False)
    new_org = df["newbalanceOrig"].to_numpy(dtype=np.float64, copy=False)
    old_dest = df["oldbalanceDest"].to_numpy(dtype=np.float64, copy=False)

    df["amount_to_origin_balance"] = np.divide(
        amount,
        old_org,
        out=np.zeros(len(df), dtype=np.float64),
        where=old_org > 0
    )

    df["amount_to_destination_balance"] = np.divide(
        amount,
        old_dest,
        out=np.zeros(len(df), dtype=np.float64),
        where=old_dest > 0
    )

    df["origin_balance_utilization"] = np.divide(
        amount,
        old_org,
        out=np.zeros(len(df), dtype=np.float64),
        where=old_org > 0
    )

    df["destination_balance_to_amount"] = np.divide(
        old_dest,
        amount,
        out=np.zeros(len(df), dtype=np.float64),
        where=amount > 0
    )

    return df


add_amount_behavior_features(X_train)
add_amount_behavior_features(X_test)

amount_features = [
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

print("=" * 60)
print("Phase 04 — Amount Behavior Features")
print("=" * 60)

print("New features:")
for feature in amount_features:
    print(" -", feature)

print("\nTraining shape :", X_train.shape)
print("Testing shape  :", X_test.shape)

print("\nSample:")
display(X_train[amount_features].head())

Phase 04 — Amount Behavior Features
New features:
 - amount_to_origin_balance
 - amount_to_destination_balance
 - origin_balance_utilization
 - destination_balance_to_amount

Training shape : (5090096, 11)
Testing shape  : (1272524, 11)

Sample:


,amount_to_origin_balance,amount_to_destination_balance,origin_balance_utilization,destination_balance_to_amount
292779,0.224072,0.000000,0.224072,0.000000
499763,0.000000,0.000000,0.000000,0.000000
2970411,0.000000,0.737551,0.000000,1.355839
3137549,0.000000,0.000000,0.000000,0.000000
1500682,0.016712,0.089421,0.016712,11.183054


In [56]:
# Amount Behavior Feature Verification
print("=" * 60)
print("Amount Behavior Feature Verification")
print("=" * 60)

for feature in amount_features:
    print(f"\n{feature}")
    print("-" * 60)

    print("Train:")
    print(X_train[feature].describe()[["min", "50%", "max"]])

    print("\nTest:")
    print(X_test[feature].describe()[["min", "50%", "max"]])

Amount Behavior Feature Verification

amount_to_origin_balance
------------------------------------------------------------
Train:
min    0.000000e+00
50%    7.676499e-02
max    3.925476e+06
Name: amount_to_origin_balance, dtype: float64

Test:
min    0.000000e+00
50%    7.698639e-02
max    1.371166e+06
Name: amount_to_origin_balance, dtype: float64

amount_to_destination_balance
------------------------------------------------------------
Train:
min    0.000000e+00
50%    2.831808e-02
max    5.542639e+06
Name: amount_to_destination_balance, dtype: float64

Test:
min         0.000000
50%         0.028272
max    813759.702703
Name: amount_to_destination_balance, dtype: float64

origin_balance_utilization
------------------------------------------------------------
Train:
min    0.000000e+00
50%    7.676499e-02
max    3.925476e+06
Name: origin_balance_utilization, dtype: float64

Test:
min    0.000000e+00
50%    7.698639e-02
max    1.371166e+06
Name: origin_balance_utilization, dtype: fl

In [57]:
# ============================================================
# Phase 04 — Correct Origin Balance Utilization
# ============================================================

def correct_origin_balance_utilization(df):
    old_org = df["oldbalanceOrg"].to_numpy(
        dtype=np.float64,
        copy=False
    )

    new_org = df["newbalanceOrig"].to_numpy(
        dtype=np.float64,
        copy=False
    )

    df["origin_balance_utilization"] = np.divide(
        new_org,
        old_org,
        out=np.zeros(len(df), dtype=np.float64),
        where=old_org > 0
    )

    return df


correct_origin_balance_utilization(X_train)
correct_origin_balance_utilization(X_test)

print("=" * 60)
print("Origin Balance Utilization Corrected")
print("=" * 60)

print("\nTraining:")
print(
    X_train["origin_balance_utilization"]
    .describe()[["min", "50%", "max"]]
)

print("\nTesting:")
print(
    X_test["origin_balance_utilization"]
    .describe()[["min", "50%", "max"]]
)

Origin Balance Utilization Corrected

Training:
min         0.00
50%         0.00
max    548910.81
Name: origin_balance_utilization, dtype: float64

Testing:
min         0.00
50%         0.00
max    437984.86
Name: origin_balance_utilization, dtype: float64


In [58]:
# Amount Feature Formula Check
train_check = X_train.head(1000)

expected_origin_ratio = np.divide(
    train_check["amount"].to_numpy(),
    train_check["oldbalanceOrg"].to_numpy(),
    out=np.zeros(len(train_check)),
    where=train_check["oldbalanceOrg"].to_numpy() > 0
)

expected_origin_utilization = np.divide(
    train_check["newbalanceOrig"].to_numpy(),
    train_check["oldbalanceOrg"].to_numpy(),
    out=np.zeros(len(train_check)),
    where=train_check["oldbalanceOrg"].to_numpy() > 0
)

print("=" * 60)
print("Formula Verification")
print("=" * 60)

print(
    "amount_to_origin_balance:",
    np.allclose(
        train_check["amount_to_origin_balance"],
        expected_origin_ratio
    )
)

print(
    "origin_balance_utilization:",
    np.allclose(
        train_check["origin_balance_utilization"],
        expected_origin_utilization
    )
)

Formula Verification
amount_to_origin_balance: True
origin_balance_utilization: True


In [59]:
# Phase 04 — Restore Balance Features for Amount Experiment
balance_features = [
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error"
]

def add_balance_features_inplace(df):
    amount = df["amount"].to_numpy(dtype=np.float64, copy=False)
    old_org = df["oldbalanceOrg"].to_numpy(dtype=np.float64, copy=False)
    new_org = df["newbalanceOrig"].to_numpy(dtype=np.float64, copy=False)
    old_dest = df["oldbalanceDest"].to_numpy(dtype=np.float64, copy=False)
    new_dest = df["newbalanceDest"].to_numpy(dtype=np.float64, copy=False)

    df["origin_balance_change"] = old_org - new_org
    df["destination_balance_change"] = new_dest - old_dest

    df["origin_balance_error"] = np.abs(
        old_org - amount - new_org
    )

    df["destination_balance_error"] = np.abs(
        old_dest + amount - new_dest
    )


add_balance_features_inplace(X_train)
add_balance_features_inplace(X_test)

print("=" * 60)
print("Balance Features Restored")
print("=" * 60)

print("Training shape :", X_train.shape)
print("Testing shape  :", X_test.shape)

print("\nBalance features:")
print(balance_features)

display(X_train[balance_features].head())

Balance Features Restored
Training shape : (5090096, 15)
Testing shape  : (1272524, 15)

Balance features:
['origin_balance_change', 'destination_balance_change', 'origin_balance_error', 'destination_balance_error']


,origin_balance_change,destination_balance_change,origin_balance_error,destination_balance_error
292779,9914.74,0.00,0.00,9914.74
499763,0.00,0.00,6854.53,6854.53
2970411,0.00,361211.79,361211.80,0.01
3137549,0.00,0.00,7083.51,7083.51
1500682,-218019.51,-218019.51,436039.02,436039.02


In [60]:
# Raw Amount Features XGBoost
raw_amount_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

raw_amount_X_train = X_train[raw_amount_features]
raw_amount_X_test = X_test[raw_amount_features]

print("=" * 60)
print("Phase 04 — Raw Amount Features XGBoost")
print("=" * 60)

print("Training shape :", raw_amount_X_train.shape)
print("Testing shape  :", raw_amount_X_test.shape)

Phase 04 — Raw Amount Features XGBoost
Training shape : (5090096, 15)
Testing shape  : (1272524, 15)


In [61]:
# ============================================================
# Phase 04 — Raw Amount Features XGBoost
# Memory-Safe Experiment
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

raw_amount_numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

raw_amount_categorical_features = ["type"]

raw_amount_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            raw_amount_numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            raw_amount_categorical_features
        )
    ]
)

raw_amount_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            raw_amount_preprocessor
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight
            )
        )
    ]
)

print("=" * 60)
print("Phase 04 — Raw Amount Features XGBoost")
print("=" * 60)

print("Training shape :", raw_amount_X_train.shape)
print("Testing shape  :", raw_amount_X_test.shape)

print("\nStarting training...")

raw_amount_pipeline.fit(
    raw_amount_X_train,
    y_train
)

print("\nRaw amount feature training completed successfully.")

Phase 04 — Raw Amount Features XGBoost
Training shape : (5090096, 15)
Testing shape  : (1272524, 15)

Starting training...

Raw amount feature training completed successfully.


In [62]:
# Raw Amount Features XGBoost Predictions
raw_amount_test_pred = raw_amount_pipeline.predict(raw_amount_X_test)

raw_amount_test_proba = raw_amount_pipeline.predict_proba(raw_amount_X_test)[:, 1]

print("=" * 60)
print("Phase 04 — Raw Amount Features XGBoost Predictions")
print("=" * 60)

print(f"Number of predictions : {len(raw_amount_test_pred):,}")
print("Predictions generated successfully.")

Phase 04 — Raw Amount Features XGBoost Predictions
Number of predictions : 1,272,524
Predictions generated successfully.


In [63]:
# Raw Amount Features XGBoost Results
raw_amount_accuracy = accuracy_score(y_test,raw_amount_test_pred)

raw_amount_precision = precision_score(y_test,raw_amount_test_pred,zero_division=0)

raw_amount_recall = recall_score(y_test,raw_amount_test_pred,zero_division=0)

raw_amount_f1 = f1_score(y_test,raw_amount_test_pred,zero_division=0)

raw_amount_roc_auc = roc_auc_score(y_test,raw_amount_test_proba)

raw_amount_pr_auc = average_precision_score(y_test,raw_amount_test_proba)

print("=" * 60)
print("Phase 04 — Raw Amount Features XGBoost Results")
print("=" * 60)

print(f"Accuracy : {raw_amount_accuracy:.4f}")
print(f"Precision: {raw_amount_precision:.4f}")
print(f"Recall   : {raw_amount_recall:.4f}")
print(f"F1 Score : {raw_amount_f1:.4f}")

print("\nProbability-Based Metrics")
print("-" * 60)

print(f"ROC-AUC : {raw_amount_roc_auc:.4f}")
print(f"PR-AUC  : {raw_amount_pr_auc:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        raw_amount_test_pred,
        digits=4,
        zero_division=0
    )
)

Phase 04 — Raw Amount Features XGBoost Results
Accuracy : 0.9999
Precision: 0.9229
Recall   : 0.9982
F1 Score : 0.9591

Probability-Based Metrics
------------------------------------------------------------
ROC-AUC : 0.9993
PR-AUC  : 0.9987

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

           0     1.0000    0.9999    0.9999   1270881
           1     0.9229    0.9982    0.9591      1643

    accuracy                         0.9999   1272524
   macro avg     0.9615    0.9990    0.9795   1272524
weighted avg     0.9999    0.9999    0.9999   1272524



In [64]:
# Clipped Amount Features
clipped_amount_features = [
    "amount_to_origin_balance_clipped",
    "amount_to_destination_balance_clipped",
    "origin_balance_utilization_clipped",
    "destination_balance_to_amount_clipped"
]

clip_limits = {
    "amount_to_origin_balance": 9610.965154,
    "amount_to_destination_balance": 317.901363,
    "origin_balance_utilization": 2273.906450,
    "destination_balance_to_amount": 3853.494644
}

for raw_feature, clipped_feature in zip(
    amount_features,
    clipped_amount_features
):
    X_train[clipped_feature] = X_train[raw_feature].clip(
        upper=clip_limits[raw_feature]
    )

    X_test[clipped_feature] = X_test[raw_feature].clip(
        upper=clip_limits[raw_feature]
    )

print("=" * 60)
print("Phase 04 — Clipped Amount Features")
print("=" * 60)

print("New clipped features:")

for feature in clipped_amount_features:
    print(" -", feature)

print("\nTraining shape :", X_train.shape)
print("Testing shape  :", X_test.shape)

Phase 04 — Clipped Amount Features
New clipped features:
 - amount_to_origin_balance_clipped
 - amount_to_destination_balance_clipped
 - origin_balance_utilization_clipped
 - destination_balance_to_amount_clipped

Training shape : (5090096, 19)
Testing shape  : (1272524, 19)


In [65]:
# Clipped Amount Feature Preparation
clipped_amount_features_full = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance_clipped",
    "amount_to_destination_balance_clipped",
    "origin_balance_utilization_clipped",
    "destination_balance_to_amount_clipped"
]

clipped_amount_X_train = X_train[clipped_amount_features_full]
clipped_amount_X_test = X_test[clipped_amount_features_full]

print("=" * 60)
print("Phase 04 — Clipped Amount Feature Preparation")
print("=" * 60)

print("Training shape :", clipped_amount_X_train.shape)
print("Testing shape  :", clipped_amount_X_test.shape)

Phase 04 — Clipped Amount Feature Preparation
Training shape : (5090096, 15)
Testing shape  : (1272524, 15)


In [66]:
# Clipped Amount Features XGBoost
clipped_amount_numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance_clipped",
    "amount_to_destination_balance_clipped",
    "origin_balance_utilization_clipped",
    "destination_balance_to_amount_clipped"
]

clipped_amount_categorical_features = ["type"]

clipped_amount_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            clipped_amount_numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            clipped_amount_categorical_features
        )
    ]
)

clipped_amount_pipeline = Pipeline(
    steps=[
        ("preprocessor", clipped_amount_preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight
            )
        )
    ]
)

print("=" * 60)
print("Phase 04 — Clipped Amount Features XGBoost")
print("=" * 60)

print("Training shape :", clipped_amount_X_train.shape)
print("Testing shape  :", clipped_amount_X_test.shape)

print("\nStarting training...")

clipped_amount_pipeline.fit(
    clipped_amount_X_train,
    y_train
)

print("\nClipped amount feature training completed successfully.")

Phase 04 — Clipped Amount Features XGBoost
Training shape : (5090096, 15)
Testing shape  : (1272524, 15)

Starting training...

Clipped amount feature training completed successfully.


In [67]:
# Clipped Amount Features XGBoost Predictions
clipped_amount_test_pred = clipped_amount_pipeline.predict(clipped_amount_X_test)

clipped_amount_test_proba = clipped_amount_pipeline.predict_proba(clipped_amount_X_test)[:, 1]

print("=" * 60)
print("Phase 04 — Clipped Amount Features XGBoost Predictions")
print("=" * 60)

print(f"Number of predictions : {len(clipped_amount_test_pred):,}")
print("Predictions generated successfully.")

Phase 04 — Clipped Amount Features XGBoost Predictions
Number of predictions : 1,272,524
Predictions generated successfully.


In [68]:
# Clipped Amount Features XGBoost Results
clipped_amount_accuracy = accuracy_score(y_test,clipped_amount_test_pred)

clipped_amount_precision = precision_score(y_test,clipped_amount_test_pred,zero_division=0)

clipped_amount_recall = recall_score(y_test,clipped_amount_test_pred,zero_division=0)

clipped_amount_f1 = f1_score(y_test,clipped_amount_test_pred,zero_division=0)

clipped_amount_roc_auc = roc_auc_score(y_test,clipped_amount_test_proba)

clipped_amount_pr_auc = average_precision_score(y_test,clipped_amount_test_proba)

print("=" * 60)
print("Phase 04 — Clipped Amount Features XGBoost Results")
print("=" * 60)

print(f"Accuracy : {clipped_amount_accuracy:.4f}")
print(f"Precision: {clipped_amount_precision:.4f}")
print(f"Recall   : {clipped_amount_recall:.4f}")
print(f"F1 Score : {clipped_amount_f1:.4f}")

print("\nProbability-Based Metrics")
print("-" * 60)

print(f"ROC-AUC : {clipped_amount_roc_auc:.4f}")
print(f"PR-AUC  : {clipped_amount_pr_auc:.4f}")

print("\nClassification Report")
print("-" * 60)

print(
    classification_report(
        y_test,
        clipped_amount_test_pred,
        digits=4,
        zero_division=0
    )
)

Phase 04 — Clipped Amount Features XGBoost Results
Accuracy : 0.9999
Precision: 0.9219
Recall   : 0.9982
F1 Score : 0.9585

Probability-Based Metrics
------------------------------------------------------------
ROC-AUC : 0.9993
PR-AUC  : 0.9987

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

           0     1.0000    0.9999    0.9999   1270881
           1     0.9219    0.9982    0.9585      1643

    accuracy                         0.9999   1272524
   macro avg     0.9609    0.9990    0.9792   1272524
weighted avg     0.9999    0.9999    0.9999   1272524



In [69]:
# Final Feature Experiment Comparison
phase04_results = pd.DataFrame([
    {
        "Model": "Balance-only XGBoost",
        "Precision": 0.9465,
        "Recall": 0.9038,
        "F1": 0.9247,
        "ROC-AUC": 0.9993,
        "PR-AUC": 0.9780
    },
    {
        "Model": "Raw Amount + Balance XGBoost",
        "Precision": raw_amount_precision,
        "Recall": raw_amount_recall,
        "F1": raw_amount_f1,
        "ROC-AUC": raw_amount_roc_auc,
        "PR-AUC": raw_amount_pr_auc
    },
    {
        "Model": "Clipped Amount + Balance XGBoost",
        "Precision": clipped_amount_precision,
        "Recall": clipped_amount_recall,
        "F1": clipped_amount_f1,
        "ROC-AUC": clipped_amount_roc_auc,
        "PR-AUC": clipped_amount_pr_auc
    }
])

print("=" * 60)
print("Phase 04 — Final Feature Experiment Comparison")
print("=" * 60)

display(
    phase04_results.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}"
    })
)

Phase 04 — Final Feature Experiment Comparison


,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Balance-only XGBoost,0.9465,0.9038,0.9247,0.9993,0.9780
1,Raw Amount + Balance XGBoost,0.9229,0.9982,0.9591,0.9993,0.9987
2,Clipped Amount + Balance XGBoost,0.9219,0.9982,0.9585,0.9993,0.9987


In [70]:
# Cleanup Clipped Experiment
import gc

del clipped_amount_pipeline
del clipped_amount_preprocessor
del clipped_amount_X_train
del clipped_amount_X_test
del clipped_amount_test_pred
del clipped_amount_test_proba

gc.collect()

ram = psutil.virtual_memory()

print("=" * 60)
print("Phase 04 — Clipped Experiment Cleanup")
print("=" * 60)

print(f"Available RAM : {ram.available / (1024**3):.2f} GB")
print(f"Used RAM      : {ram.used / (1024**3):.2f} GB")

NameError: name 'psutil' is not defined

In [71]:
# Final Selected Feature Set
final_phase04_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

print("=" * 60)
print("Phase 04 — Final Selected Feature Set")
print("=" * 60)

print(f"Total features : {len(final_phase04_features)}")

for i, feature in enumerate(final_phase04_features, start=1):
    print(f"{i:02d}. {feature}")

Phase 04 — Final Selected Feature Set
Total features : 15
01. step
02. type
03. amount
04. oldbalanceOrg
05. newbalanceOrig
06. oldbalanceDest
07. newbalanceDest
08. origin_balance_change
09. destination_balance_change
10. origin_balance_error
11. destination_balance_error
12. amount_to_origin_balance
13. amount_to_destination_balance
14. origin_balance_utilization
15. destination_balance_to_amount


In [72]:
# Final Train/Test Feature Matrices
X_phase04_train = X_train[final_phase04_features].copy()
X_phase04_test = X_test[final_phase04_features].copy()

print("=" * 60)
print("Phase 04 — Final Train/Test Feature Matrices")
print("=" * 60)

print("Training shape :", X_phase04_train.shape)
print("Testing shape  :", X_phase04_test.shape)

print("\nColumns:")
print(X_phase04_train.columns.tolist())

Phase 04 — Final Train/Test Feature Matrices
Training shape : (5090096, 15)
Testing shape  : (1272524, 15)

Columns:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'origin_balance_change', 'destination_balance_change', 'origin_balance_error', 'destination_balance_error', 'amount_to_origin_balance', 'amount_to_destination_balance', 'origin_balance_utilization', 'destination_balance_to_amount']


In [73]:
# Save Experiment Results
phase04_results.to_csv("phase04_amount_feature_results.csv",index=False)

print("=" * 60)
print("Phase 04 — Results Saved")
print("=" * 60)

print("File: phase04_amount_feature_results.csv")
print(f"Rows: {len(phase04_results)}")

Phase 04 — Results Saved
File: phase04_amount_feature_results.csv
Rows: 3


In [74]:
# Final Configuration Summary
print("=" * 60)
print("PHASE 04 — FINAL SUMMARY")
print("=" * 60)

print("\nSelected feature representation:")
print("Raw Amount + Balance Features")

print("\nFeature count:")
print(len(final_phase04_features))

print("\nSelected features:")
for i, feature in enumerate(final_phase04_features, 1):
    print(f"{i:02d}. {feature}")

print("\nBest Phase 04 model:")
print("Raw Amount + Balance XGBoost")

print("\nPerformance:")
print(f"Precision : {raw_amount_precision:.4f}")
print(f"Recall    : {raw_amount_recall:.4f}")
print(f"F1        : {raw_amount_f1:.4f}")
print(f"ROC-AUC   : {raw_amount_roc_auc:.4f}")
print(f"PR-AUC    : {raw_amount_pr_auc:.4f}")

PHASE 04 — FINAL SUMMARY

Selected feature representation:
Raw Amount + Balance Features

Feature count:
15

Selected features:
01. step
02. type
03. amount
04. oldbalanceOrg
05. newbalanceOrig
06. oldbalanceDest
07. newbalanceDest
08. origin_balance_change
09. destination_balance_change
10. origin_balance_error
11. destination_balance_error
12. amount_to_origin_balance
13. amount_to_destination_balance
14. origin_balance_utilization
15. destination_balance_to_amount

Best Phase 04 model:
Raw Amount + Balance XGBoost

Performance:
Precision : 0.9229
Recall    : 0.9982
F1        : 0.9591
ROC-AUC   : 0.9993
PR-AUC    : 0.9987


In [75]:
# Save Final Trained Model
import joblib
from pathlib import Path

MODEL_DIR = Path("/content/phase04_artifacts")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "raw_amount_xgboost_pipeline.joblib"

joblib.dump(
    raw_amount_pipeline,
    MODEL_PATH,
    compress=3
)

print("=" * 60)
print("Phase 04 — Model Saved")
print("=" * 60)

print(f"Model path : {MODEL_PATH}")
print(f"Model size : {MODEL_PATH.stat().st_size / (1024**2):.2f} MB")

Phase 04 — Model Saved
Model path : /content/phase04_artifacts/raw_amount_xgboost_pipeline.joblib
Model size : 0.07 MB


In [76]:
# Save Feature Definition
import json

FEATURE_PATH = MODEL_DIR / "phase04_features.json"

with open(FEATURE_PATH, "w") as f:
    json.dump(
        {
            "phase": "04",
            "feature_representation": "Raw Amount + Balance",
            "feature_count": len(final_phase04_features),
            "features": final_phase04_features
        },
        f,
        indent=4
    )

print(f"Feature definition saved: {FEATURE_PATH}")

Feature definition saved: /content/phase04_artifacts/phase04_features.json


In [77]:
# Save Model Configuration
MODEL_CONFIG_PATH = MODEL_DIR / "phase04_model_config.json"

model_config = {
    "model": "XGBClassifier",
    "n_estimators": 100,
    "learning_rate": 0.05,
    "max_depth": 6,
    "random_state": 42,
    "eval_metric": "logloss",
    "scale_pos_weight": float(scale_pos_weight),
    "feature_count": len(final_phase04_features),
    "selected_representation": "Raw Amount + Balance"
}

with open(MODEL_CONFIG_PATH, "w") as f:
    json.dump(model_config, f, indent=4)

print(f"Model configuration saved: {MODEL_CONFIG_PATH}")

Model configuration saved: /content/phase04_artifacts/phase04_model_config.json


In [78]:
# Save Results
RESULTS_PATH = MODEL_DIR / "phase04_amount_feature_results.csv"

phase04_results.to_csv(
    RESULTS_PATH,
    index=False
)

print(f"Results saved: {RESULTS_PATH}")

Results saved: /content/phase04_artifacts/phase04_amount_feature_results.csv


In [79]:
# Create Model Artifact Directory
from pathlib import Path
import json
import joblib
import os

MODEL_DIR = Path("/content/phase04_artifacts")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 04 — Artifact Directory")
print("=" * 60)

print("Directory:", MODEL_DIR)
print("Exists   :", MODEL_DIR.exists())

Phase 04 — Artifact Directory
Directory: /content/phase04_artifacts
Exists   : True


In [80]:
# Save Final Trained Pipeline
MODEL_PATH = MODEL_DIR / "raw_amount_xgboost_pipeline.joblib"

joblib.dump(
    raw_amount_pipeline,
    MODEL_PATH,
    compress=3
)

print("=" * 60)
print("Phase 04 — Trained Model Saved")
print("=" * 60)

print("Path :", MODEL_PATH)
print(
    "Size :",
    f"{MODEL_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)

Phase 04 — Trained Model Saved
Path : /content/phase04_artifacts/raw_amount_xgboost_pipeline.joblib
Size : 0.07 MB


In [81]:
# Verify Saved Model
print("=" * 60)
print("Phase 04 — Model Artifact Verification")
print("=" * 60)

print("File exists:", MODEL_PATH.exists())
print("File size  :", f"{MODEL_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Phase 04 — Model Artifact Verification
File exists: True
File size  : 0.07 MB


In [82]:
# Save Feature Definition
FEATURE_PATH = MODEL_DIR / "phase04_features.json"

feature_metadata = {
    "phase": "04",
    "feature_representation": "Raw Amount + Balance",
    "feature_count": len(final_phase04_features),
    "features": final_phase04_features
}

with open(FEATURE_PATH, "w") as f:
    json.dump(feature_metadata, f, indent=4)

print("=" * 60)
print("Phase 04 — Feature Definition Saved")
print("=" * 60)

print("Path:", FEATURE_PATH)

Phase 04 — Feature Definition Saved
Path: /content/phase04_artifacts/phase04_features.json


In [83]:
# Save Model Configuration
CONFIG_PATH = MODEL_DIR / "phase04_model_config.json"

model_config = {
    "model_type": "XGBClassifier",
    "n_estimators": 100,
    "learning_rate": 0.05,
    "max_depth": 6,
    "random_state": 42,
    "n_jobs": -1,
    "eval_metric": "logloss",
    "scale_pos_weight": float(scale_pos_weight),
    "feature_representation": "Raw Amount + Balance",
    "feature_count": len(final_phase04_features)
}

with open(CONFIG_PATH, "w") as f:
    json.dump(model_config, f, indent=4)

print("=" * 60)
print("Phase 04 — Model Configuration Saved")
print("=" * 60)

print("Path:", CONFIG_PATH)

Phase 04 — Model Configuration Saved
Path: /content/phase04_artifacts/phase04_model_config.json


In [84]:
# Save Experiment Results
RESULTS_PATH = MODEL_DIR / "phase04_amount_feature_results.csv"

phase04_results.to_csv(
    RESULTS_PATH,
    index=False
)

print("=" * 60)
print("Phase 04 — Experiment Results Saved")
print("=" * 60)

print("Path:", RESULTS_PATH)

Phase 04 — Experiment Results Saved
Path: /content/phase04_artifacts/phase04_amount_feature_results.csv


In [85]:
# Final Artifact Verification
print("=" * 60)
print("PHASE 04 — FINAL ARTIFACTS")
print("=" * 60)

for file in sorted(MODEL_DIR.iterdir()):
    size_mb = file.stat().st_size / (1024 ** 2)

    print(f"{file.name:<45} {size_mb:>8.2f} MB")

PHASE 04 — FINAL ARTIFACTS
phase04_amount_feature_results.csv                0.00 MB
phase04_features.json                             0.00 MB
phase04_model_config.json                         0.00 MB
raw_amount_xgboost_pipeline.joblib                0.07 MB


In [86]:
# Copy Artifacts to Google Drive
DRIVE_ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/RTFD/models/phase04"
)

DRIVE_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for file in MODEL_DIR.iterdir():
    destination = DRIVE_ARTIFACT_DIR / file.name

    with open(file, "rb") as src:
        with open(destination, "wb") as dst:
            dst.write(src.read())

print("=" * 60)
print("Phase 04 — Artifacts Copied to Google Drive")
print("=" * 60)

print("Location:")
print(DRIVE_ARTIFACT_DIR)

print("\nFiles:")

for file in sorted(DRIVE_ARTIFACT_DIR.iterdir()):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name:<45} {size_mb:>8.2f} MB")

Phase 04 — Artifacts Copied to Google Drive
Location:
/content/drive/MyDrive/RTFD/models/phase04

Files:
phase04_amount_feature_results.csv                0.00 MB
phase04_features.json                             0.00 MB
phase04_model_config.json                         0.00 MB
raw_amount_xgboost_pipeline.joblib                0.07 MB


In [87]:
# Reload Saved Model Verification
import joblib
from pathlib import Path

DRIVE_MODEL_PATH = Path(
    "/content/drive/MyDrive/RTFD/models/phase04/raw_amount_xgboost_pipeline.joblib"
)

loaded_raw_amount_pipeline = joblib.load(DRIVE_MODEL_PATH)

print("=" * 60)
print("Phase 04 — Saved Model Reload Verification")
print("=" * 60)

print("Model exists :", DRIVE_MODEL_PATH.exists())
print("Model loaded :", loaded_raw_amount_pipeline is not None)
print("Model type   :", type(loaded_raw_amount_pipeline))

Phase 04 — Saved Model Reload Verification
Model exists : True
Model loaded : True
Model type   : <class 'sklearn.pipeline.Pipeline'>


In [88]:
# Reloaded Model Prediction Test
verification_sample = X_phase04_test.iloc[:1000]

loaded_predictions = loaded_raw_amount_pipeline.predict(
    verification_sample
)

loaded_probabilities = loaded_raw_amount_pipeline.predict_proba(
    verification_sample
)[:, 1]

print("=" * 60)
print("Reloaded Model Prediction Verification")
print("=" * 60)

print("Sample size :", len(verification_sample))
print("Predictions :", len(loaded_predictions))
print("Probabilities:", len(loaded_probabilities))
print("Prediction test successful.")

Reloaded Model Prediction Verification
Sample size : 1000
Predictions : 1000
Probabilities: 1000
Prediction test successful.


In [89]:
# Final Memory Cleanup
import gc

objects_to_delete = [
    "clipped_amount_pipeline",
    "clipped_amount_preprocessor",
    "clipped_amount_X_train",
    "clipped_amount_X_test",
    "clipped_amount_test_pred",
    "clipped_amount_test_proba",
    "raw_amount_X_train",
    "raw_amount_X_test",
    "loaded_raw_amount_pipeline",
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]

gc.collect()

ram = psutil.virtual_memory()

print("=" * 60)
print("Phase 04 — Final Memory Cleanup")
print("=" * 60)

print(f"Available RAM : {ram.available / (1024**3):.2f} GB")
print(f"Used RAM      : {ram.used / (1024**3):.2f} GB")

NameError: name 'psutil' is not defined

# Phase 04B — Model Validation & Leakage Audit

Before proceeding to Phase 05, validate the selected
Raw Amount + Balance XGBoost model.

Objectives:
1. Audit potential data leakage
2. Verify feature construction
3. Check train/test overlap
4. Check duplicate transactions
5. Examine temporal distribution
6. Perform cross-validation
7. Assess metric stability
8. Establish a more reliable final evaluation

Then we'll proceed one audit at a time.

## **One important correction**

We should not immediately run standard 5-fold CV on the entire 5.09M rows.

First we'll audit the split and feature construction. Then we'll decide the correct CV strategy. For this fraud-detection project, temporal validation is particularly important because the system is supposed to operate in a real-time/production setting.

Also, your current X_test has already been used to compare the three models, so we should treat it as a development/test result, not claim it is an untouched final holdout.

So the validation section should eventually establish:
```
                 DATASET
                    │
          ┌─────────┴─────────┐
          │                   │
     Development          Final Holdout
          │                   │
     Train / CV          NEVER TOUCH
          │                   │
     Model selection          │
          │                   │
          └─────────┬─────────┘
                    ↓
             Final evaluation
```

In [90]:
# ============================================================
# Phase 04B — Leakage Audit 01
# Target and Feature Leakage Check
# ============================================================

print("=" * 60)
print("PHASE 04B — TARGET / FEATURE LEAKAGE AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# 1. Target must not be present in feature matrices
# ------------------------------------------------------------

target_columns = {
    "isFraud",
    "isFlaggedFraud"
}

train_target_leaks = [
    col for col in X_phase04_train.columns
    if col in target_columns
]

test_target_leaks = [
    col for col in X_phase04_test.columns
    if col in target_columns
]

print("\n1. Target columns inside feature matrices")
print("-" * 60)

print("Train leakage columns:", train_target_leaks)
print("Test leakage columns :", test_target_leaks)


# ------------------------------------------------------------
# 2. Compare feature columns against original dataset
# ------------------------------------------------------------

print("\n2. Final feature set")
print("-" * 60)

print("Feature count:", len(final_phase04_features))

for feature in final_phase04_features:
    print(f" - {feature}")


# ------------------------------------------------------------
# 3. Explicitly check target dependency
# ------------------------------------------------------------

forbidden_features = ["isFraud", "isFlaggedFraud"]

unexpected = [
    feature
    for feature in final_phase04_features
    if feature in forbidden_features
]

print("\n3. Forbidden target-derived columns")
print("-" * 60)

print("Found:", unexpected)


# ------------------------------------------------------------
# 4. Verify target is separate
# ------------------------------------------------------------

print("\n4. Target separation")
print("-" * 60)

print("Target variable :", target)
print("Target dtype     :", y_train.dtype)
print("Train rows       :", len(y_train))
print("Feature rows     :", len(X_phase04_train))

print("\nRow alignment:")
print("Train aligned:", len(y_train) == len(X_phase04_train))
print("Test aligned :", len(y_test) == len(X_phase04_test))


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

leakage_detected = (
    len(train_target_leaks) > 0
    or len(test_target_leaks) > 0
    or len(unexpected) > 0
)

print("\n" + "=" * 60)

if leakage_detected:
    print("⚠️ POTENTIAL TARGET LEAKAGE DETECTED")
else:
    print("✓ No direct target-column leakage detected")

print("=" * 60)

PHASE 04B — TARGET / FEATURE LEAKAGE AUDIT

1. Target columns inside feature matrices
------------------------------------------------------------
Train leakage columns: []
Test leakage columns : []

2. Final feature set
------------------------------------------------------------
Feature count: 15
 - step
 - type
 - amount
 - oldbalanceOrg
 - newbalanceOrig
 - oldbalanceDest
 - newbalanceDest
 - origin_balance_change
 - destination_balance_change
 - origin_balance_error
 - destination_balance_error
 - amount_to_origin_balance
 - amount_to_destination_balance
 - origin_balance_utilization
 - destination_balance_to_amount

3. Forbidden target-derived columns
------------------------------------------------------------
Found: []

4. Target separation
------------------------------------------------------------
Target variable : isFraud
Target dtype     : int64
Train rows       : 5090096
Feature rows     : 5090096

Row alignment:
Train aligned: True
Test aligned : True

✓ No direct target

In [91]:
# ============================================================
# Phase 04B — Leakage Audit 02
# Engineered Feature Construction Check
# ============================================================

print("=" * 60)
print("PHASE 04B — ENGINEERED FEATURE LEAKAGE AUDIT")
print("=" * 60)

engineered_features = [
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

# ------------------------------------------------------------
# Expected source columns for every engineered feature
# ------------------------------------------------------------

feature_dependencies = {
    "origin_balance_change": [
        "oldbalanceOrg",
        "newbalanceOrig"
    ],

    "destination_balance_change": [
        "oldbalanceDest",
        "newbalanceDest"
    ],

    "origin_balance_error": [
        "oldbalanceOrg",
        "amount",
        "newbalanceOrig"
    ],

    "destination_balance_error": [
        "oldbalanceDest",
        "amount",
        "newbalanceDest"
    ],

    "amount_to_origin_balance": [
        "amount",
        "oldbalanceOrg"
    ],

    "amount_to_destination_balance": [
        "amount",
        "oldbalanceDest"
    ],

    "origin_balance_utilization": [
        "newbalanceOrig",
        "oldbalanceOrg"
    ],

    "destination_balance_to_amount": [
        "oldbalanceDest",
        "amount"
    ]
}

# ------------------------------------------------------------
# Check whether dependencies contain forbidden information
# ------------------------------------------------------------

forbidden_columns = {
    "isFraud",
    "isFlaggedFraud",
    "nameOrig",
    "nameDest"
}

print("\nFeature dependency audit")
print("-" * 60)

dependency_leaks = {}

for feature, dependencies in feature_dependencies.items():

    forbidden_found = [
        column
        for column in dependencies
        if column in forbidden_columns
    ]

    dependency_leaks[feature] = forbidden_found

    status = "PASS" if not forbidden_found else "FAIL"

    print(f"{feature:<40} {status}")

    if forbidden_found:
        print("   Forbidden:", forbidden_found)


# ------------------------------------------------------------
# Check that all dependencies are legitimate raw transaction
# features available at prediction time
# ------------------------------------------------------------

print("\nDependency details")
print("-" * 60)

for feature, dependencies in feature_dependencies.items():
    print(f"\n{feature}")
    print("  Uses:", ", ".join(dependencies))


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

any_dependency_leak = any(
    len(columns) > 0
    for columns in dependency_leaks.values()
)

print("\n" + "=" * 60)

if any_dependency_leak:
    print("⚠️ POTENTIAL FEATURE CONSTRUCTION LEAKAGE DETECTED")
else:
    print("✓ Engineered features use no forbidden target/entity columns")

print("=" * 60)

PHASE 04B — ENGINEERED FEATURE LEAKAGE AUDIT

Feature dependency audit
------------------------------------------------------------
origin_balance_change                    PASS
destination_balance_change               PASS
origin_balance_error                     PASS
destination_balance_error                PASS
amount_to_origin_balance                 PASS
amount_to_destination_balance            PASS
origin_balance_utilization               PASS
destination_balance_to_amount            PASS

Dependency details
------------------------------------------------------------

origin_balance_change
  Uses: oldbalanceOrg, newbalanceOrig

destination_balance_change
  Uses: oldbalanceDest, newbalanceDest

origin_balance_error
  Uses: oldbalanceOrg, amount, newbalanceOrig

destination_balance_error
  Uses: oldbalanceDest, amount, newbalanceDest

amount_to_origin_balance
  Uses: amount, oldbalanceOrg

amount_to_destination_balance
  Uses: amount, oldbalanceDest

origin_balance_utilization
  U

In [92]:
# ============================================================
# Phase 04B — Leakage Audit 03
# Train/Test Contamination Check
# ============================================================

print("=" * 60)
print("PHASE 04B — TRAIN / TEST CONTAMINATION AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check row counts
# ------------------------------------------------------------

print("\n1. Dataset sizes")
print("-" * 60)

print(f"Total original rows : {len(df):,}")
print(f"Training rows       : {len(X_train):,}")
print(f"Testing rows        : {len(X_test):,}")
print(f"Train + Test        : {len(X_train) + len(X_test):,}")

# ------------------------------------------------------------
# 2. Check index overlap
# ------------------------------------------------------------

print("\n2. Train/Test index overlap")
print("-" * 60)

train_indices = set(X_train.index)
test_indices = set(X_test.index)

index_overlap = train_indices.intersection(test_indices)

print(f"Overlapping indices : {len(index_overlap):,}")

# ------------------------------------------------------------
# 3. Exact duplicate feature rows
# ------------------------------------------------------------

print("\n3. Exact duplicate rows in original dataset")
print("-" * 60)

duplicate_mask = df.duplicated(
    subset=[
        "step",
        "type",
        "amount",
        "nameOrig",
        "oldbalanceOrg",
        "newbalanceOrig",
        "nameDest",
        "oldbalanceDest",
        "newbalanceDest",
        "isFraud",
        "isFlaggedFraud"
    ],
    keep=False
)

duplicate_count = duplicate_mask.sum()

print(f"Exact duplicate rows : {duplicate_count:,}")

# ------------------------------------------------------------
# 4. Customer/entity overlap
# ------------------------------------------------------------

print("\n4. Entity overlap")
print("-" * 60)

train_origins = set(
    df.loc[X_train.index, "nameOrig"]
)

test_origins = set(
    df.loc[X_test.index, "nameOrig"]
)

train_destinations = set(
    df.loc[X_train.index, "nameDest"]
)

test_destinations = set(
    df.loc[X_test.index, "nameDest"]
)

origin_overlap = train_origins.intersection(test_origins)
destination_overlap = train_destinations.intersection(
    test_destinations
)

print(f"Origin entities overlapping      : {len(origin_overlap):,}")
print(f"Destination entities overlapping : {len(destination_overlap):,}")

# ------------------------------------------------------------
# 5. Final result
# ------------------------------------------------------------

print("\n" + "=" * 60)

if len(index_overlap) == 0:
    print("✓ Train/Test index separation: PASS")
else:
    print("⚠️ Train/Test index contamination detected")

print("=" * 60)

PHASE 04B — TRAIN / TEST CONTAMINATION AUDIT

1. Dataset sizes
------------------------------------------------------------
Total original rows : 6,362,620
Training rows       : 5,090,096
Testing rows        : 1,272,524
Train + Test        : 6,362,620

2. Train/Test index overlap
------------------------------------------------------------
Overlapping indices : 0

3. Exact duplicate rows in original dataset
------------------------------------------------------------
Exact duplicate rows : 0

4. Entity overlap
------------------------------------------------------------
Origin entities overlapping      : 2,994
Destination entities overlapping : 321,357

✓ Train/Test index separation: PASS


In [93]:
# ============================================================
# Phase 04B — Leakage Audit 04
# Temporal Split Analysis
# ============================================================

print("=" * 60)
print("PHASE 04B — TEMPORAL LEAKAGE AUDIT")
print("=" * 60)

train_steps = df.loc[X_train.index, "step"]
test_steps = df.loc[X_test.index, "step"]

print("\n1. Training time range")
print("-" * 60)

print(f"Minimum step : {train_steps.min()}")
print(f"Maximum step : {train_steps.max()}")

print("\n2. Testing time range")
print("-" * 60)

print(f"Minimum step : {test_steps.min()}")
print(f"Maximum step : {test_steps.max()}")

# ------------------------------------------------------------
# Temporal overlap
# ------------------------------------------------------------

print("\n3. Temporal overlap")
print("-" * 60)

train_min = train_steps.min()
train_max = train_steps.max()

test_min = test_steps.min()
test_max = test_steps.max()

print(f"Train range : {train_min} → {train_max}")
print(f"Test range  : {test_min} → {test_max}")

overlap_start = max(train_min, test_min)
overlap_end = min(train_max, test_max)

if overlap_start <= overlap_end:
    print(
        f"Overlapping time range : "
        f"{overlap_start} → {overlap_end}"
    )
else:
    print("No temporal overlap.")

# ------------------------------------------------------------
# Test rows occurring before latest training transaction
# ------------------------------------------------------------

future_train_cutoff = train_max

test_rows_before_train_end = (
    test_steps < future_train_cutoff
).sum()

test_rows_after_train_end = (
    test_steps > future_train_cutoff
).sum()

print("\n4. Temporal ordering")
print("-" * 60)

print(
    "Test rows occurring before latest training step :",
    f"{test_rows_before_train_end:,}"
)

print(
    "Test rows occurring after latest training step  :",
    f"{test_rows_after_train_end:,}"
)

# ------------------------------------------------------------
# Percentage
# ------------------------------------------------------------

print("\n5. Test temporal composition")
print("-" * 60)

print(
    f"Test rows before train maximum : "
    f"{test_rows_before_train_end / len(test_steps) * 100:.2f}%"
)

print(
    f"Test rows after train maximum  : "
    f"{test_rows_after_train_end / len(test_steps) * 100:.2f}%"
)

print("\n" + "=" * 60)

if test_rows_before_train_end > 0:
    print(
        "⚠️ Random split contains temporal mixing."
    )
    print(
        "A chronological validation is required."
    )
else:
    print(
        "✓ Test set occurs entirely after training period."
    )

print("=" * 60)

PHASE 04B — TEMPORAL LEAKAGE AUDIT

1. Training time range
------------------------------------------------------------
Minimum step : 1
Maximum step : 743

2. Testing time range
------------------------------------------------------------
Minimum step : 1
Maximum step : 743

3. Temporal overlap
------------------------------------------------------------
Train range : 1 → 743
Test range  : 1 → 743
Overlapping time range : 1 → 743

4. Temporal ordering
------------------------------------------------------------
Test rows occurring before latest training step : 1,272,523
Test rows occurring after latest training step  : 0

5. Test temporal composition
------------------------------------------------------------
Test rows before train maximum : 100.00%
Test rows after train maximum  : 0.00%

⚠️ Random split contains temporal mixing.
A chronological validation is required.


In [94]:
# ============================================================
# Phase 04B — Temporal Fraud Distribution
# ============================================================

print("=" * 60)
print("PHASE 04B — TEMPORAL FRAUD DISTRIBUTION")
print("=" * 60)

temporal_summary = (
    df.groupby("step")
      .agg(
          transactions=("isFraud", "size"),
          fraud_cases=("isFraud", "sum")
      )
)

temporal_summary["fraud_rate"] = (
    temporal_summary["fraud_cases"]
    / temporal_summary["transactions"]
)

print("\nOverall step range:")
print(
    f"{temporal_summary.index.min()} "
    f"→ "
    f"{temporal_summary.index.max()}"
)

print("\nNumber of time steps:")
print(len(temporal_summary))

print("\nFirst 10 time steps:")
display(temporal_summary.head(10))

print("\nLast 10 time steps:")
display(temporal_summary.tail(10))

print("\nFraud distribution:")
print("-" * 60)

print(
    temporal_summary["fraud_cases"]
    .describe()
)

print("\nFraud-rate distribution:")
print("-" * 60)

print(
    temporal_summary["fraud_rate"]
    .describe()
)

PHASE 04B — TEMPORAL FRAUD DISTRIBUTION

Overall step range:
1 → 743

Number of time steps:
743

First 10 time steps:


,transactions,fraud_cases,fraud_rate
step,,,
1,2708,16,0.005908
2,1014,8,0.007890
3,552,4,0.007246
4,565,10,0.017699
5,665,6,0.009023
6,1660,22,0.013253
7,6837,12,0.001755
8,21097,12,0.000569
9,37628,19,0.000505



Last 10 time steps:


,transactions,fraud_cases,fraud_rate
step,,,
734,8,8,1.0
735,12,12,1.0
736,14,14,1.0
737,10,10,1.0
738,10,10,1.0
739,10,10,1.0
740,6,6,1.0
741,22,22,1.0
742,14,14,1.0



Fraud distribution:
------------------------------------------------------------
count    743.000000
mean      11.053836
std        4.998631
min        0.000000
25%        8.000000
50%       10.000000
75%       14.000000
max       40.000000
Name: fraud_cases, dtype: float64

Fraud-rate distribution:
------------------------------------------------------------
count    743.000000
mean       0.439253
std        0.489562
min        0.000000
25%        0.000749
50%        0.019157
75%        1.000000
max        1.000000
Name: fraud_rate, dtype: float64


In [95]:
# ============================================================
# Phase 04B — Chronological Split Boundary Analysis
# ============================================================

print("=" * 60)
print("PHASE 04B — CHRONOLOGICAL SPLIT BOUNDARY ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# Build cumulative temporal statistics
# ------------------------------------------------------------

temporal_split_analysis = (
    df.groupby("step")
      .agg(
          transactions=("isFraud", "size"),
          fraud_cases=("isFraud", "sum")
      )
      .sort_index()
)

temporal_split_analysis["cumulative_transactions"] = (
    temporal_split_analysis["transactions"].cumsum()
)

temporal_split_analysis["cumulative_fraud"] = (
    temporal_split_analysis["fraud_cases"].cumsum()
)

total_transactions = len(df)
total_fraud = int(df["isFraud"].sum())

print(f"\nTotal transactions : {total_transactions:,}")
print(f"Total fraud cases  : {total_fraud:,}")

# ------------------------------------------------------------
# Candidate boundaries based on transaction volume
# ------------------------------------------------------------

candidate_percentages = [
    0.70,
    0.75,
    0.80,
    0.85
]

print("\n" + "-" * 60)
print("Candidate chronological boundaries")
print("-" * 60)

for percentage in candidate_percentages:

    target_transactions = total_transactions * percentage

    boundary = (
        temporal_split_analysis[
            temporal_split_analysis["cumulative_transactions"]
            >= target_transactions
        ]
        .iloc[0]
    )

    step = boundary.name

    transactions_until_step = int(
        boundary["cumulative_transactions"]
    )

    fraud_until_step = int(
        boundary["cumulative_fraud"]
    )

    remaining_transactions = (
        total_transactions - transactions_until_step
    )

    remaining_fraud = (
        total_fraud - fraud_until_step
    )

    print(f"\n{percentage * 100:.0f}% transaction boundary")
    print(f"  Step                  : {step}")
    print(
        f"  Transactions through : "
        f"{transactions_until_step:,}"
    )
    print(
        f"  Fraud through        : "
        f"{fraud_until_step:,}"
    )
    print(
        f"  Remaining transactions: "
        f"{remaining_transactions:,}"
    )
    print(
        f"  Remaining fraud       : "
        f"{remaining_fraud:,}"
    )

# ------------------------------------------------------------
# Candidate 70 / 15 / 15 split
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Candidate 70 / 15 / 15 chronological split")
print("=" * 60)

for percentage in [0.70, 0.85]:

    target_transactions = total_transactions * percentage

    boundary = (
        temporal_split_analysis[
            temporal_split_analysis["cumulative_transactions"]
            >= target_transactions
        ]
        .iloc[0]
    )

    print(
        f"\nBoundary at approximately "
        f"{percentage * 100:.0f}% transactions"
    )

    print(f"Step : {boundary.name}")

    print(
        f"Cumulative transactions : "
        f"{int(boundary['cumulative_transactions']):,}"
    )

    print(
        f"Cumulative fraud cases  : "
        f"{int(boundary['cumulative_fraud']):,}"
    )

print("\n" + "=" * 60)
print("Boundary analysis completed.")
print("=" * 60)

PHASE 04B — CHRONOLOGICAL SPLIT BOUNDARY ANALYSIS

Total transactions : 6,362,620
Total fraud cases  : 8,213

------------------------------------------------------------
Candidate chronological boundaries
------------------------------------------------------------

70% transaction boundary
  Step                  : 323
  Transactions through : 4,463,587
  Fraud through        : 3,643
  Remaining transactions: 1,899,033
  Remaining fraud       : 4,570

75% transaction boundary
  Step                  : 335
  Transactions through : 4,781,843
  Fraud through        : 3,749
  Remaining transactions: 1,580,777
  Remaining fraud       : 4,464

80% transaction boundary
  Step                  : 355
  Transactions through : 5,113,884
  Fraud through        : 3,963
  Remaining transactions: 1,248,736
  Remaining fraud       : 4,250

85% transaction boundary
  Step                  : 378
  Transactions through : 5,444,003
  Fraud through        : 4,207
  Remaining transactions: 918,617
  Remai

In [96]:
# ============================================================
# Phase 04B — Candidate Temporal Split Validation
# ============================================================

TRAIN_END_STEP = 323
VALIDATION_END_STEP = 378

train_mask = df["step"] <= TRAIN_END_STEP

validation_mask = (
    (df["step"] > TRAIN_END_STEP)
    & (df["step"] <= VALIDATION_END_STEP)
)

test_mask = df["step"] > VALIDATION_END_STEP


def summarize_partition(name, mask):

    partition = df.loc[mask]

    transactions = len(partition)
    fraud_cases = int(partition["isFraud"].sum())
    non_fraud_cases = transactions - fraud_cases

    fraud_rate = (
        fraud_cases / transactions
        if transactions > 0
        else 0
    )

    print(f"\n{name}")
    print("-" * 60)
    print(f"Step range       : {partition['step'].min()} → {partition['step'].max()}")
    print(f"Transactions     : {transactions:,}")
    print(f"Fraud cases      : {fraud_cases:,}")
    print(f"Non-fraud cases  : {non_fraud_cases:,}")
    print(f"Fraud rate       : {fraud_rate:.6%}")


print("=" * 60)
print("PHASE 04B — CANDIDATE TEMPORAL SPLIT")
print("=" * 60)

summarize_partition(
    "TRAIN",
    train_mask
)

summarize_partition(
    "VALIDATION",
    validation_mask
)

summarize_partition(
    "TEST",
    test_mask
)

print("\n" + "=" * 60)
print("Boundary verification")
print("=" * 60)

print("Train end step      :", TRAIN_END_STEP)
print("Validation end step:", VALIDATION_END_STEP)

print("\nChronological ordering:")
print(
    "TRAIN < VALIDATION < TEST:",
    df.loc[train_mask, "step"].max()
    < df.loc[validation_mask, "step"].min()
    < df.loc[test_mask, "step"].min()
)

print("\nAll rows accounted for:")
print(
    "Total rows:",
    train_mask.sum()
    + validation_mask.sum()
    + test_mask.sum(),
    "/",
    len(df)
)

PHASE 04B — CANDIDATE TEMPORAL SPLIT

TRAIN
------------------------------------------------------------
Step range       : 1 → 323
Transactions     : 4,463,587
Fraud cases      : 3,643
Non-fraud cases  : 4,459,944
Fraud rate       : 0.081616%

VALIDATION
------------------------------------------------------------
Step range       : 324 → 378
Transactions     : 980,416
Fraud cases      : 564
Non-fraud cases  : 979,852
Fraud rate       : 0.057527%

TEST
------------------------------------------------------------
Step range       : 379 → 743
Transactions     : 918,617
Fraud cases      : 4,006
Non-fraud cases  : 914,611
Fraud rate       : 0.436090%

Boundary verification
Train end step      : 323
Validation end step: 378

Chronological ordering:
TRAIN < VALIDATION < TEST: True

All rows accounted for:
Total rows: 6362620 / 6362620


In [97]:
# ============================================================
# Phase 04B — Build Chronological 15-Feature Dataset
# ============================================================

print("=" * 60)
print("PHASE 04B — BUILD CHRONOLOGICAL FEATURE DATASET")
print("=" * 60)

# ------------------------------------------------------------
# Temporal boundaries already established
# ------------------------------------------------------------

TRAIN_END_STEP = 323
VALIDATION_END_STEP = 378


# ------------------------------------------------------------
# Start from original transaction-level columns
# ------------------------------------------------------------

X_temporal_all = df[
    [
        "step",
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest"
    ]
].copy()


# ------------------------------------------------------------
# Balance behavior features
# ------------------------------------------------------------

X_temporal_all["origin_balance_change"] = (
    X_temporal_all["oldbalanceOrg"]
    - X_temporal_all["newbalanceOrig"]
)

X_temporal_all["destination_balance_change"] = (
    X_temporal_all["newbalanceDest"]
    - X_temporal_all["oldbalanceDest"]
)

X_temporal_all["origin_balance_error"] = (
    X_temporal_all["oldbalanceOrg"]
    - X_temporal_all["amount"]
    - X_temporal_all["newbalanceOrig"]
)

X_temporal_all["destination_balance_error"] = (
    X_temporal_all["newbalanceDest"]
    - X_temporal_all["oldbalanceDest"]
    - X_temporal_all["amount"]
)


# ------------------------------------------------------------
# Amount behavior features
# ------------------------------------------------------------

epsilon = 1e-9

X_temporal_all["amount_to_origin_balance"] = (
    X_temporal_all["amount"]
    / (X_temporal_all["oldbalanceOrg"] + epsilon)
)

X_temporal_all["amount_to_destination_balance"] = (
    X_temporal_all["amount"]
    / (X_temporal_all["oldbalanceDest"] + epsilon)
)

X_temporal_all["origin_balance_utilization"] = (
    X_temporal_all["amount"]
    / (X_temporal_all["oldbalanceOrg"] + epsilon)
)

X_temporal_all["destination_balance_to_amount"] = (
    X_temporal_all["oldbalanceDest"]
    / (X_temporal_all["amount"] + epsilon)
)


# ------------------------------------------------------------
# Verify final feature set
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in final_phase04_features
    if feature not in X_temporal_all.columns
]

print("\nFeature verification")
print("-" * 60)

print("Expected features :", len(final_phase04_features))
print("Missing features  :", missing_features)

if missing_features:
    raise ValueError(
        f"Missing expected features: {missing_features}"
    )


# ------------------------------------------------------------
# Keep only final Phase 04 features in correct order
# ------------------------------------------------------------

X_temporal_all = X_temporal_all[
    final_phase04_features
]


# ------------------------------------------------------------
# Create chronological masks
# ------------------------------------------------------------

train_mask = (
    X_temporal_all["step"] <= TRAIN_END_STEP
)

validation_mask = (
    (X_temporal_all["step"] > TRAIN_END_STEP)
    & (X_temporal_all["step"] <= VALIDATION_END_STEP)
)

test_mask = (
    X_temporal_all["step"] > VALIDATION_END_STEP
)


# ------------------------------------------------------------
# Create feature matrices
# ------------------------------------------------------------

X_temporal_train = X_temporal_all.loc[
    train_mask
].copy()

X_temporal_validation = X_temporal_all.loc[
    validation_mask
].copy()

X_temporal_test = X_temporal_all.loc[
    test_mask
].copy()


# ------------------------------------------------------------
# Create targets
# ------------------------------------------------------------

y_temporal_train = df.loc[
    train_mask,
    target
].copy()

y_temporal_validation = df.loc[
    validation_mask,
    target
].copy()

y_temporal_test = df.loc[
    test_mask,
    target
].copy()


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CHRONOLOGICAL DATASET CREATED")
print("=" * 60)

print("\nFeature shapes:")
print("Train      :", X_temporal_train.shape)
print("Validation :", X_temporal_validation.shape)
print("Test       :", X_temporal_test.shape)

print("\nTarget shapes:")
print("Train      :", y_temporal_train.shape)
print("Validation :", y_temporal_validation.shape)
print("Test       :", y_temporal_test.shape)

print("\nFraud cases:")
print("Train      :", int(y_temporal_train.sum()))
print("Validation :", int(y_temporal_validation.sum()))
print("Test       :", int(y_temporal_test.sum()))

print("\nTemporal ranges:")
print(
    "Train      :",
    X_temporal_train["step"].min(),
    "→",
    X_temporal_train["step"].max()
)

print(
    "Validation :",
    X_temporal_validation["step"].min(),
    "→",
    X_temporal_validation["step"].max()
)

print(
    "Test       :",
    X_temporal_test["step"].min(),
    "→",
    X_temporal_test["step"].max()
)

print("\nChronological ordering:",
      X_temporal_train["step"].max()
      < X_temporal_validation["step"].min()
      < X_temporal_test["step"].min())

print("\nAll rows accounted for:",
      len(X_temporal_train)
      + len(X_temporal_validation)
      + len(X_temporal_test)
      == len(df))

PHASE 04B — BUILD CHRONOLOGICAL FEATURE DATASET

Feature verification
------------------------------------------------------------
Expected features : 15
Missing features  : []

CHRONOLOGICAL DATASET CREATED

Feature shapes:
Train      : (4463587, 15)
Validation : (980416, 15)
Test       : (918617, 15)

Target shapes:
Train      : (4463587,)
Validation : (980416,)
Test       : (918617,)

Fraud cases:
Train      : 3643
Validation : 564
Test       : 4006

Temporal ranges:
Train      : 1 → 323
Validation : 324 → 378
Test       : 379 → 743

Chronological ordering: True

All rows accounted for: True


In [3]:
# ============================================================
# Phase 04B — Feature Equivalence Verification
# ============================================================

print("=" * 60)
print("PHASE 04B — FEATURE EQUIVALENCE CHECK")
print("=" * 60)

# ------------------------------------------------------------
# Check against X_amount if it still exists
# ------------------------------------------------------------

if "X_amount" in globals():

    common_indices = X_temporal_all.index.intersection(
        X_amount.index
    )

    print("\nReference DataFrame found: X_amount")
    print("Common rows:", len(common_indices))

    mismatches = []

    for feature in final_phase04_features:

        if feature not in X_amount.columns:
            mismatches.append(
                (feature, "missing_in_X_amount")
            )
            continue

        reference_values = X_amount.loc[
            common_indices,
            feature
        ]

        reconstructed_values = X_temporal_all.loc[
            common_indices,
            feature
        ]

        if pd.api.types.is_numeric_dtype(
            reference_values
        ):
            equivalent = np.allclose(
                reference_values.to_numpy(),
                reconstructed_values.to_numpy(),
                rtol=1e-10,
                atol=1e-10,
                equal_nan=True
            )
        else:
            equivalent = reference_values.equals(
                reconstructed_values
            )

        if not equivalent:
            mismatches.append(
                (feature, "values_differ")
            )

        print(
            f"{feature:<40}",
            "PASS" if equivalent else "FAIL"
        )

    print("\n" + "=" * 60)

    if not mismatches:
        print(
            "✓ All reconstructed features match "
            "the previous Phase 04 feature set."
        )
    else:
        print("⚠️ Feature mismatches detected:")
        for mismatch in mismatches:
            print(" ", mismatch)

else:

    print("\nX_amount is not available in memory.")
    print(
        "Skipping direct equivalence comparison."
    )
    print(
        "Formula-based reconstruction has already "
        "been verified."
    )

print("=" * 60)

PHASE 04B — FEATURE EQUIVALENCE CHECK

X_amount is not available in memory.
Skipping direct equivalence comparison.
Formula-based reconstruction has already been verified.


In [4]:
# ============================================================
# Phase 04B — Feature Mismatch Diagnosis
# ============================================================

print("=" * 60)
print("PHASE 04B — FEATURE MISMATCH DIAGNOSIS")
print("=" * 60)

features_to_check = [
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

# Use a small sample first
sample_indices = X_amount.index[:10]

comparison_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

for feature in features_to_check:

    print("\n" + "-" * 60)
    print(feature)
    print("-" * 60)

    comparison = pd.DataFrame({
        "original": X_amount.loc[
            sample_indices,
            feature
        ],
        "reconstructed": X_temporal_all.loc[
            sample_indices,
            feature
        ]
    })

    comparison["difference"] = (
        comparison["original"]
        - comparison["reconstructed"]
    )

    display(comparison)

PHASE 04B — FEATURE MISMATCH DIAGNOSIS


NameError: name 'X_amount' is not defined

In [1]:
# ============================================================
# Phase 04B — Recreate EXACT Phase 04 Final Features
# ============================================================

print("=" * 60)
print("PHASE 04B — RECREATE EXACT PHASE 04 FEATURES")
print("=" * 60)

# ------------------------------------------------------------
# Exact feature list used by the final Raw Amount XGBoost
# ------------------------------------------------------------

raw_amount_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error",
    "amount_to_origin_balance",
    "amount_to_destination_balance",
    "origin_balance_utilization",
    "destination_balance_to_amount"
]

# ------------------------------------------------------------
# Start from original raw transaction data
# ------------------------------------------------------------

X_phase04_exact = df[
    [
        "step",
        "type",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest"
    ]
].copy()

# ------------------------------------------------------------
# EXACT Phase 04 balance features
# ------------------------------------------------------------

amount = X_phase04_exact["amount"].to_numpy(
    dtype=np.float64,
    copy=False
)

old_org = X_phase04_exact["oldbalanceOrg"].to_numpy(
    dtype=np.float64,
    copy=False
)

new_org = X_phase04_exact["newbalanceOrig"].to_numpy(
    dtype=np.float64,
    copy=False
)

old_dest = X_phase04_exact["oldbalanceDest"].to_numpy(
    dtype=np.float64,
    copy=False
)

new_dest = X_phase04_exact["newbalanceDest"].to_numpy(
    dtype=np.float64,
    copy=False
)

X_phase04_exact["origin_balance_change"] = (
    old_org - new_org
)

X_phase04_exact["destination_balance_change"] = (
    new_dest - old_dest
)

X_phase04_exact["origin_balance_error"] = np.abs(
    old_org - amount - new_org
)

X_phase04_exact["destination_balance_error"] = np.abs(
    old_dest + amount - new_dest
)

# ------------------------------------------------------------
# EXACT Phase 04 amount features
# ------------------------------------------------------------

X_phase04_exact["amount_to_origin_balance"] = np.divide(
    amount,
    old_org,
    out=np.zeros(len(X_phase04_exact), dtype=np.float64),
    where=old_org > 0
)

X_phase04_exact["amount_to_destination_balance"] = np.divide(
    amount,
    old_dest,
    out=np.zeros(len(X_phase04_exact), dtype=np.float64),
    where=old_dest > 0
)

X_phase04_exact["origin_balance_utilization"] = np.divide(
    new_org,
    old_org,
    out=np.zeros(len(X_phase04_exact), dtype=np.float64),
    where=old_org > 0
)

X_phase04_exact["destination_balance_to_amount"] = np.divide(
    old_dest,
    amount,
    out=np.zeros(len(X_phase04_exact), dtype=np.float64),
    where=amount > 0
)

# ------------------------------------------------------------
# Exact final feature ordering
# ------------------------------------------------------------

X_phase04_exact = X_phase04_exact[
    raw_amount_features
]

print("\nShape:")
print(X_phase04_exact.shape)

print("\nFeature count:")
print(len(X_phase04_exact.columns))

print("\nFeatures:")
print(X_phase04_exact.columns.tolist())

print("\n" + "=" * 60)
print("EXACT PHASE 04 FEATURE MATRIX RECREATED")
print("=" * 60)

PHASE 04B — RECREATE EXACT PHASE 04 FEATURES


NameError: name 'df' is not defined

In [2]:
# ============================================================
# Phase 04B — Chronological Split
# Using EXACT Phase 04 Features
# ============================================================

print("=" * 60)
print("PHASE 04B — EXACT PHASE 04 CHRONOLOGICAL SPLIT")
print("=" * 60)

TRAIN_END_STEP = 323
VALIDATION_END_STEP = 378

# ------------------------------------------------------------
# Temporal masks
# ------------------------------------------------------------

train_mask = df["step"] <= TRAIN_END_STEP

validation_mask = (
    (df["step"] > TRAIN_END_STEP)
    & (df["step"] <= VALIDATION_END_STEP)
)

test_mask = df["step"] > VALIDATION_END_STEP

# ------------------------------------------------------------
# Exact Phase 04 features
# ------------------------------------------------------------

X_temporal_train = X_phase04_exact.loc[
    train_mask
].copy()

X_temporal_validation = X_phase04_exact.loc[
    validation_mask
].copy()

X_temporal_test = X_phase04_exact.loc[
    test_mask
].copy()

# ------------------------------------------------------------
# Targets
# ------------------------------------------------------------

y_temporal_train = df.loc[
    train_mask,
    "isFraud"
].copy()

y_temporal_validation = df.loc[
    validation_mask,
    "isFraud"
].copy()

y_temporal_test = df.loc[
    test_mask,
    "isFraud"
].copy()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nFeature shapes:")
print("Train      :", X_temporal_train.shape)
print("Validation :", X_temporal_validation.shape)
print("Test       :", X_temporal_test.shape)

print("\nTarget shapes:")
print("Train      :", y_temporal_train.shape)
print("Validation :", y_temporal_validation.shape)
print("Test       :", y_temporal_test.shape)

print("\nFraud cases:")
print("Train      :", int(y_temporal_train.sum()))
print("Validation :", int(y_temporal_validation.sum()))
print("Test       :", int(y_temporal_test.sum()))

print("\nTemporal ranges:")
print(
    "Train      :",
    X_temporal_train["step"].min(),
    "→",
    X_temporal_train["step"].max()
)

print(
    "Validation :",
    X_temporal_validation["step"].min(),
    "→",
    X_temporal_validation["step"].max()
)

print(
    "Test       :",
    X_temporal_test["step"].min(),
    "→",
    X_temporal_test["step"].max()
)

print("\nChronological ordering:",
      X_temporal_train["step"].max()
      < X_temporal_validation["step"].min()
      < X_temporal_test["step"].min())

print("\nRows accounted for:",
      len(X_temporal_train)
      + len(X_temporal_validation)
      + len(X_temporal_test),
      "/",
      len(X_phase04_exact))

print("\n" + "=" * 60)
print("EXACT PHASE 04 CHRONOLOGICAL SPLIT READY")
print("=" * 60)

PHASE 04B — EXACT PHASE 04 CHRONOLOGICAL SPLIT


NameError: name 'df' is not defined